In [1]:
# Chapter 6.1: LSTM price prediction figure for April 1--15
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


savefigure = 1  # 1: save to thesis figures; 0: display only
THESIS_FIGURE_DIR = Path(r"D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures")

RUN_NAME = "madrl_train_7days"
FALLBACK_RUN_NAME = "madrl_traindays_7"
CONTROLLER_KEY = "MADRL_BASE"

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "artifacts" / "runs").exists() and (path / "AAA-Thesis").exists():
            return path
    raise FileNotFoundError("Could not locate project root from the current notebook directory.")

project_root = find_project_root(Path.cwd().resolve())
run_dir = project_root / "artifacts" / "runs" / RUN_NAME
if not run_dir.exists():
    fallback_run_dir = project_root / "artifacts" / "runs" / FALLBACK_RUN_NAME
    if fallback_run_dir.exists():
        print(f"Run '{RUN_NAME}' was not found. Using existing run '{FALLBACK_RUN_NAME}' instead.")
        run_dir = fallback_run_dir
    else:
        raise FileNotFoundError(f"Neither {run_dir} nor {fallback_run_dir} exists.")

record_path = run_dir / "results" / "lstm" / CONTROLLER_KEY / "record" / "step.parquet"
if not record_path.exists():
    candidates = sorted((run_dir / "results" / "lstm").glob("*/record/step.parquet"))
    if not candidates:
        raise FileNotFoundError(f"No LSTM step.parquet files were found under {run_dir}.")
    record_path = candidates[0]
    print(f"Controller '{CONTROLLER_KEY}' was not found. Using {record_path.parent.parent.name} instead.")

step_df = pd.read_parquet(record_path).copy()
time_col = "timestamp" if "timestamp" in step_df.columns else "step"
actual_col = "import_price" if "import_price" in step_df.columns else "wholesale_price"
pred_col = f"{actual_col}_pred"
if pred_col not in step_df.columns:
    raise KeyError(f"Expected prediction column '{pred_col}' in {record_path}.")

plot_df = step_df.sort_values(time_col).copy()
plot_df[time_col] = pd.to_datetime(plot_df[time_col])
timezone = plot_df[time_col].dt.tz
window_start = pd.Timestamp("2020-04-01", tz=timezone)
window_end = pd.Timestamp("2020-04-16", tz=timezone)
plot_df = plot_df.loc[plot_df[time_col].between(window_start, window_end, inclusive="left")].copy()
if plot_df.empty:
    raise ValueError(f"No prediction records found in [{window_start}, {window_end}).")

plot_df["price_error"] = plot_df[pred_col] - plot_df[actual_col]
mae = plot_df["price_error"].abs().mean()
rmse = np.sqrt((plot_df["price_error"] ** 2).mean())
max_abs_error = plot_df["price_error"].abs().max()

fig, axes = plt.subplots(2, 1, figsize=(12, 6.8), sharex=True, gridspec_kw={"height_ratios": [2.2, 1]})
axes[0].plot(plot_df[time_col], plot_df[actual_col], label="Actual import price", color="#1f77b4", linewidth=1.8)
axes[0].plot(plot_df[time_col], plot_df[pred_col], label="LSTM prediction", color="#d62728", linewidth=1.6, linestyle="--")
axes[0].set_ylabel("Price (EUR/kWh)")
axes[0].legend(loc="best", frameon=False)
axes[0].grid(True, alpha=0.25)

axes[1].plot(plot_df[time_col], plot_df["price_error"], label="Prediction error", color="#2ca02c", linewidth=1.4)
axes[1].axhline(0.0, color="black", linewidth=0.8, alpha=0.6)
axes[1].set_ylabel("Error")
axes[1].set_xlabel("Time")
axes[1].grid(True, alpha=0.25)
axes[1].text(
    0.01,
    0.92,
    f"MAE={mae:.4f}, RMSE={rmse:.4f}, MaxAE={max_abs_error:.4f}",
    transform=axes[1].transAxes,
    va="top",
    fontsize=10,
)

fig.autofmt_xdate()
fig.tight_layout()

if savefigure == 1:
    THESIS_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    output_path = THESIS_FIGURE_DIR / "ch6_1_lstm_price_prediction_madrl_train_7days.png"
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"Saved figure to: {output_path}")
else:
    print("savefigure=0: figure is displayed in the notebook output and was not saved.")

plt.show()


Run 'madrl_train_7days' was not found. Using existing run 'madrl_traindays_7' instead.


Saved figure to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures\ch6_1_lstm_price_prediction_madrl_train_7days.png


C:\Users\20539\AppData\Local\Temp\ipykernel_21208\2977018675.py:94: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [2]:
# Chapter 6.2: MADRL convergence figure using Base_soft reward curves
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


savefigure = 1  # 1: save to thesis figures; 0: display only
THESIS_FIGURE_DIR = Path(r"D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures")

RUN_SPECS = [
    {
        "run_name": "20260607_235607_100",
        "label": "Base_soft, 100 episodes",
        "expected_episodes": 100,
        "color": "#1f77b4",
    },
    {
        "run_name": "madrl_traindays_7",
        "label": "Base_soft, 1000 episodes",
        "expected_episodes": 1000,
        "color": "#d62728",
    },
]

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "artifacts" / "runs").exists() and (path / "AAA-Thesis").exists():
            return path
    raise FileNotFoundError("Could not locate project root from the current notebook directory.")

def load_reward_curve(project_root: Path, run_name: str, expected_episodes: int) -> pd.DataFrame:
    reward_path = project_root / "artifacts" / "runs" / run_name / "tables" / "madrl_reward_curves_madrl_base.csv"
    if not reward_path.exists():
        raise FileNotFoundError(f"Reward curve file was not found: {reward_path}")
    df = pd.read_csv(reward_path).sort_values("episode").copy()
    episode_min = int(df["episode"].min())
    episode_max = int(df["episode"].max())
    episode_count = int(df["episode"].nunique())
    print(f"{run_name}: episode range {episode_min}-{episode_max}, unique episodes={episode_count}")
    if episode_count != expected_episodes or episode_max != expected_episodes:
        raise ValueError(
            f"{run_name} was expected to contain {expected_episodes} episodes, "
            f"but got {episode_count} unique episodes and max episode {episode_max}."
        )
    return df

def add_rolling_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    window = 10 if df["episode"].max() <= 100 else 50
    df["reward_smooth"] = df["total_reward"].rolling(window=window, min_periods=1).mean()
    if "critic_loss" in df.columns:
        df["critic_loss_smooth"] = df["critic_loss"].rolling(window=window, min_periods=1).mean()
    else:
        df["critic_loss_smooth"] = pd.NA
    df["rolling_window"] = window
    return df

project_root = find_project_root(Path.cwd().resolve())
curves = []
for spec in RUN_SPECS:
    curve = load_reward_curve(project_root, spec["run_name"], spec["expected_episodes"])
    curve = add_rolling_columns(curve)
    curve["run_name"] = spec["run_name"]
    curve["plot_label"] = spec["label"]
    curve["plot_color"] = spec["color"]
    curves.append(curve)

fig, axes = plt.subplots(2, 1, figsize=(12, 7.2), sharex=False, gridspec_kw={"height_ratios": [2.2, 1.2]})

for curve in curves:
    label = curve["plot_label"].iloc[0]
    color = curve["plot_color"].iloc[0]
    window = int(curve["rolling_window"].iloc[0])
    axes[0].plot(curve["episode"], curve["total_reward"], color=color, alpha=0.18, linewidth=0.8)
    axes[0].plot(curve["episode"], curve["reward_smooth"], color=color, linewidth=2.0, label=f"{label} ({window}-episode mean)")
    valid_loss = curve.dropna(subset=["critic_loss_smooth"])
    if not valid_loss.empty:
        axes[1].plot(valid_loss["episode"], valid_loss["critic_loss_smooth"], color=color, linewidth=1.8, label=label)

axes[0].set_title("Base_soft Training Convergence")
axes[0].set_ylabel("Total reward")
axes[0].grid(True, alpha=0.25)
axes[0].legend(loc="best", frameon=False)

axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Critic loss")
axes[1].grid(True, alpha=0.25)
axes[1].legend(loc="best", frameon=False)

fig.tight_layout()

if savefigure == 1:
    THESIS_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    output_path = THESIS_FIGURE_DIR / "ch6_2_madrl_convergence_madrl_base_100_vs_1000.png"
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"Saved figure to: {output_path}")
else:
    print("savefigure=0: figure is displayed in the notebook output and was not saved.")

plt.show()


20260607_235607_100: episode range 1-100, unique episodes=100
madrl_traindays_7: episode range 1-1000, unique episodes=1000


Saved figure to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures\ch6_2_madrl_convergence_madrl_base_100_vs_1000.png


C:\Users\20539\AppData\Local\Temp\ipykernel_21208\815564401.py:101: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
# Chapter 6.3: MADRL soft-control postprocessing for the 7-day training run
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print


savefigure = 1  # 1: save figures and tables; 0: display only
THESIS_FIGURE_DIR = Path(r"D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures")
RUN_NAME = "madrl_traindays_7"
DT_HOURS = 0.25
EV_DEPARTURE_STEP = 28
EV_REQUIRED_SOC = 0.90
EV_COST_WEIGHT_FACTORS = {"MADRL_BASE_COSTWEIGHT": 1.5}

MADRL_SOFT_SPECS = [
    {"key": "MADRL_BASE", "folder": "lstm", "label": "Base_soft", "stage": "Base_soft", "color": "#4C78A8"},
    {"key": "MADRL_BASE_COSTWEIGHT", "folder": "lstm", "label": "Base_Costweight_soft", "stage": "Base_Costweight_soft", "color": "#F58518"},
    {"key": "MADRL_BASE_PROGRESS_CONTINUOUS", "folder": "lstm", "label": "Base_Progress_soft", "stage": "Base_Progress_soft", "color": "#54A24B"},
    {"key": "madrl_projection_safe_Continuous_Progress", "folder": "lstm", "label": "Projection_Progress_soft", "stage": "Projection_Progress_soft", "color": "#B279A2"},
    {"key": "MADRL_PROJECTION_EV_EME", "folder": "lstm", "label": "Projection_EV_EME_soft", "stage": "Projection_EV_EME_soft", "color": "#E45756"},
    {"key": "MADRL_BASE_EV_EME", "folder": "lstm", "label": "Base_EME_soft", "stage": "Base_EME_soft", "color": "#72B7B2"},
]

BASELINE_SPECS = [
    {"key": "admm_mpc_lstm_EV_soft", "folder": "lstm", "label": "ADMM-MPC Soft", "stage": "Baseline", "color": "#9D755D"},
    {"key": "local_mpc_lstm_EV_soft", "folder": "lstm", "label": "Local MPC Soft", "stage": "Baseline", "color": "#BAB0AC"},
    {"key": "rule_based_ev_with_battery", "folder": "perfect", "label": "Rule-based with Battery", "stage": "Baseline", "color": "#8CD17D"},
]

ALL_SPECS = MADRL_SOFT_SPECS + BASELINE_SPECS

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "artifacts" / "runs").exists() and (path / "AAA-Thesis").exists():
            return path
    raise FileNotFoundError("Could not locate project root from the current notebook directory.")

def read_records(project_root: Path, spec: dict) -> dict:
    record_dir = project_root / "artifacts" / "runs" / RUN_NAME / "results" / spec["folder"] / spec["key"] / "record"
    required_files = ["metrics.parquet", "step.parquet", "agent.parquet"]
    missing = [name for name in required_files if not (record_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing {missing} under {record_dir}")
    return {
        "spec": spec,
        "metrics": pd.read_parquet(record_dir / "metrics.parquet"),
        "step": pd.read_parquet(record_dir / "step.parquet"),
        "agent": pd.read_parquet(record_dir / "agent.parquet"),
    }

def series_sum(df: pd.DataFrame, col: str) -> float:
    return float(df[col].sum()) if col in df.columns else 0.0

def metric_value(metrics: pd.Series, col: str, default=np.nan) -> float:
    return float(metrics[col]) if col in metrics.index and pd.notna(metrics[col]) else default

def summarize_controller(record: dict) -> dict:
    spec = record["spec"]
    metrics = record["metrics"].iloc[0]
    step = record["step"]
    agent = record["agent"]
    departure = agent.loc[agent["step"] == EV_DEPARTURE_STEP].copy()
    departure_gap = departure["ev_departure_gap"] if "ev_departure_gap" in departure.columns else pd.Series(dtype=float)
    ev_soc = departure["ev_soc"] if "ev_soc" in departure.columns else pd.Series(dtype=float)
    raw_ev_charging_cost = series_sum(step, "ev_charging_cost_eur")
    ev_cost_weight = EV_COST_WEIGHT_FACTORS.get(spec["key"], 1.0)
    actual_ev_charging_cost = raw_ev_charging_cost / ev_cost_weight
    raw_total_eur = metric_value(metrics, "total_eur")
    actual_total_eur = raw_total_eur - raw_ev_charging_cost + actual_ev_charging_cost
    return {
        "label": spec["label"],
        "key": spec["key"],
        "group": "MADRL Soft" if spec in MADRL_SOFT_SPECS else "Baseline",
        "stage": spec["stage"],
        "color": spec["color"],
        "total_eur": actual_total_eur,
        "raw_total_eur": raw_total_eur,
        "system_other_cost_eur": metric_value(metrics, "system_other_cost_eur"),
        "storage_profit_total_eur": metric_value(metrics, "storage_profit_total_eur"),
        "storage_charge_cost_total_eur": metric_value(metrics, "storage_charge_cost_total_eur"),
        "storage_discharge_revenue_total_eur": metric_value(metrics, "storage_discharge_revenue_total_eur"),
        "ev_charging_cost_eur": actual_ev_charging_cost,
        "raw_ev_charging_cost_eur": raw_ev_charging_cost,
        "ev_cost_weight_factor": ev_cost_weight,
        "mean_departure_soc": float(ev_soc.mean()) if len(ev_soc) else np.nan,
        "min_departure_soc": float(ev_soc.min()) if len(ev_soc) else np.nan,
        "departure_soc_below_req_count": int((ev_soc < EV_REQUIRED_SOC - 1e-9).sum()) if len(ev_soc) else 0,
        "departure_gap_total_soc": float(departure_gap.sum()) if len(departure_gap) else 0.0,
        "progress_gap_total_soc": series_sum(agent, "ev_progress_gap"),
        "price_aware_penalty_total": series_sum(agent, "ev_price_aware_penalty"),
        "emergency_added_kwh": series_sum(agent, "ev_emergency_added_kw") * DT_HOURS,
        "projection_gap_kwh": series_sum(agent, "ev_projection_gap_kw") * DT_HOURS,
        "voltage_violation_count": metric_value(metrics, "voltage_violation_count", 0.0),
        "voltage_violation_steps": metric_value(metrics, "voltage_violation_steps", 0.0),
        "min_vm_pu": metric_value(metrics, "min_vm_pu"),
        "max_vm_pu": metric_value(metrics, "max_vm_pu"),
        "trafo_overload_steps": metric_value(metrics, "trafo_overload_steps", 0.0),
        "trafo_loading_max_pct": metric_value(metrics, "trafo_loading_max_pct"),
        "feeder_netload_ramp_mean_abs_kw": metric_value(metrics, "feeder_netload_ramp_mean_abs_kw"),
        "battery_net_power_kw_mean_abs": metric_value(metrics, "battery_net_power_kw_mean_abs"),
    }

def build_price_bin_table(record: dict) -> dict:
    spec = record["spec"]
    step = record["step"][["episode_idx", "step", "import_price"]].copy()
    agent = record["agent"].merge(step, on=["episode_idx", "step"], how="left")
    agent = agent.loc[agent.get("ev_available", 1) == 1].copy()
    if agent.empty or "ev_charge_kw" not in agent.columns:
        return {"label": spec["label"], "weighted_charge_price_eur_per_kwh": np.nan, "low_price_energy_kwh": 0.0, "mid_price_energy_kwh": 0.0, "high_price_energy_kwh": 0.0, "low_price_share": 0.0, "high_price_share": 0.0}
    q_low, q_high = step["import_price"].quantile([1 / 3, 2 / 3])
    agent["ev_energy_kwh"] = agent["ev_charge_kw"].clip(lower=0.0) * DT_HOURS
    agent["price_bin"] = pd.cut(agent["import_price"], bins=[-np.inf, q_low, q_high, np.inf], labels=["low", "mid", "high"])
    energy_by_bin = agent.groupby("price_bin", observed=False)["ev_energy_kwh"].sum()
    total_energy = float(agent["ev_energy_kwh"].sum())
    weighted_price = float((agent["ev_energy_kwh"] * agent["import_price"]).sum() / total_energy) if total_energy > 0 else np.nan
    return {
        "label": spec["label"],
        "weighted_charge_price_eur_per_kwh": weighted_price,
        "total_ev_energy_kwh": total_energy,
        "low_price_energy_kwh": float(energy_by_bin.get("low", 0.0)),
        "mid_price_energy_kwh": float(energy_by_bin.get("mid", 0.0)),
        "high_price_energy_kwh": float(energy_by_bin.get("high", 0.0)),
        "low_price_share": float(energy_by_bin.get("low", 0.0) / total_energy) if total_energy > 0 else 0.0,
        "high_price_share": float(energy_by_bin.get("high", 0.0) / total_energy) if total_energy > 0 else 0.0,
    }

project_root = find_project_root(Path.cwd().resolve())
records = [read_records(project_root, spec) for spec in ALL_SPECS]
summary_df = pd.DataFrame([summarize_controller(record) for record in records])
price_bin_df = pd.DataFrame([build_price_bin_table(record) for record in records])

economic_table = summary_df[["label", "group", "total_eur", "system_other_cost_eur", "storage_profit_total_eur", "ev_charging_cost_eur"]].round(3)
ev_departure_table = summary_df[["label", "mean_departure_soc", "min_departure_soc", "departure_soc_below_req_count", "departure_gap_total_soc", "progress_gap_total_soc", "emergency_added_kwh", "projection_gap_kwh"]].round(4)
grid_safety_table = summary_df[["label", "voltage_violation_count", "voltage_violation_steps", "min_vm_pu", "max_vm_pu", "trafo_overload_steps", "trafo_loading_max_pct"]].round(4)
price_table = price_bin_df.round(4)

print("Economic metrics")
display(economic_table)
print("EV departure SoC and EV feasibility metrics")
display(ev_departure_table)
print("Grid safety metrics")
display(grid_safety_table)
print("EV charging and electricity-price relationship")
display(price_table)

madrl_summary = summary_df.loc[summary_df["group"] == "MADRL Soft"].copy()
all_colors = summary_df.set_index("label")["color"].to_dict()
figures = []

fig1, ax1 = plt.subplots(figsize=(11.5, 4.8))
ax1.bar(madrl_summary["stage"], madrl_summary["total_eur"], color=madrl_summary["color"])
ax1.axhline(0.0, color="black", linewidth=0.8)
ax1.set_title("MADRL Soft Logic Chain: Total Cost")
ax1.set_ylabel("Total cost (EUR)")
ax1.tick_params(axis="x", rotation=25)
ax1.grid(True, axis="y", alpha=0.25)
fig1.tight_layout()
figures.append((fig1, "ch6_3_soft_logic_chain_total_cost.png"))

fig2, ax2 = plt.subplots(figsize=(12, 5.2))
ax2.bar(summary_df["label"], summary_df["total_eur"], color=[all_colors[x] for x in summary_df["label"]])
ax2.axhline(0.0, color="black", linewidth=0.8)
ax2.set_title("MADRL Soft Controllers and Baselines: Total Cost")
ax2.set_ylabel("Total cost (EUR)")
ax2.tick_params(axis="x", rotation=35)
ax2.grid(True, axis="y", alpha=0.25)
fig2.tight_layout()
figures.append((fig2, "ch6_3_soft_controller_total_cost_comparison.png"))

fig3, axes3 = plt.subplots(1, 2, figsize=(13, 4.8))
axes3[0].bar(summary_df["label"], summary_df["voltage_violation_count"], color=[all_colors[x] for x in summary_df["label"]])
axes3[0].set_title("Voltage Violation Count")
axes3[0].set_ylabel("Count")
axes3[0].tick_params(axis="x", rotation=40)
axes3[0].grid(True, axis="y", alpha=0.25)
axes3[1].bar(summary_df["label"], summary_df["trafo_loading_max_pct"], color=[all_colors[x] for x in summary_df["label"]])
axes3[1].axhline(100.0, color="black", linewidth=0.9, linestyle="--")
axes3[1].set_title("Maximum Transformer Loading")
axes3[1].set_ylabel("Loading (%)")
axes3[1].tick_params(axis="x", rotation=40)
axes3[1].grid(True, axis="y", alpha=0.25)
fig3.tight_layout()
figures.append((fig3, "ch6_3_soft_grid_safety_comparison.png"))

episode_to_plot = 0
plot_specs = MADRL_SOFT_SPECS
timestamp_tz = pd.to_datetime(records[0]["agent"]["timestamp"]).dt.tz
ev_window_start = pd.Timestamp("2020-04-02 18:00", tz=timestamp_tz)
ev_window_end = pd.Timestamp("2020-04-03 07:00", tz=timestamp_tz)
fig4, ax4 = plt.subplots(figsize=(12, 5.4))
price_axis = ax4.twinx()
for record in records[:len(MADRL_SOFT_SPECS)]:
    spec = record["spec"]
    agent = record["agent"]
    agent_time = pd.to_datetime(agent["timestamp"])
    ev_window_agent = agent.loc[agent_time.between(ev_window_start, ev_window_end, inclusive="both")].copy()
    ev_charge = ev_window_agent.groupby("timestamp")["ev_charge_kw"].sum().sort_index()
    ax4.plot(ev_charge.index, ev_charge.values, label=spec["label"], color=spec["color"], linewidth=1.5)
base_step_all = records[0]["step"].copy()
base_step_time = pd.to_datetime(base_step_all["timestamp"])
ev_window_step = base_step_all.loc[base_step_time.between(ev_window_start, ev_window_end, inclusive="both")].sort_values("timestamp")
price_axis.plot(ev_window_step["timestamp"], ev_window_step["import_price"], color="black", linewidth=1.2, linestyle="--", label="Import price")
ax4.set_ylabel("Total EV charging power (kW)")
price_axis.set_ylabel("Import price (EUR/kWh)")
ax4.set_xlabel("Time")
ax4.grid(True, alpha=0.25)
lines, labels = ax4.get_legend_handles_labels()
lines2, labels2 = price_axis.get_legend_handles_labels()
ax4.legend(lines + lines2, labels + labels2, loc="upper left", frameon=False, ncol=2)
fig4.autofmt_xdate()
fig4.tight_layout()
figures.append((fig4, "ch6_3_soft_ev_charge_price_episode0.png"))

base_step = records[0]["step"].loc[records[0]["step"]["episode_idx"] == episode_to_plot].sort_values("timestamp")
fig5, ax5 = plt.subplots(figsize=(12, 5.4))
price_axis5 = ax5.twinx()
for record in records[:len(MADRL_SOFT_SPECS)]:
    spec = record["spec"]
    agent = record["agent"]
    day_agent = agent.loc[agent["episode_idx"] == episode_to_plot].copy()
    battery_power = day_agent.groupby("timestamp")["e_bat"].sum().sort_index()
    ax5.plot(battery_power.index, battery_power.values, label=spec["label"], color=spec["color"], linewidth=1.5)
price_axis5.plot(base_step["timestamp"], base_step["import_price"], color="black", linewidth=1.2, linestyle="--", label="Import price")
ax5.axhline(0.0, color="black", linewidth=0.8, alpha=0.5)
ax5.set_ylabel("Total battery power (kW, + charge / - discharge)")
price_axis5.set_ylabel("Import price (EUR/kWh)")
ax5.set_xlabel("Time")
ax5.grid(True, alpha=0.25)
lines, labels = ax5.get_legend_handles_labels()
lines2, labels2 = price_axis5.get_legend_handles_labels()
ax5.legend(lines + lines2, labels + labels2, loc="upper left", frameon=False, ncol=2)
fig5.autofmt_xdate()
fig5.tight_layout()
figures.append((fig5, "ch6_3_soft_battery_power_price_episode0.png"))

AGENT_DETAIL_CONTROLLER_KEY = "MADRL_BASE_PROGRESS_CONTINUOUS"
detail_record = next(record for record in records if record["spec"]["key"] == AGENT_DETAIL_CONTROLLER_KEY)
detail_agent = detail_record["agent"].copy()
detail_step = detail_record["step"].sort_values("timestamp")
agent_ids = sorted(detail_agent["agent_id"].unique())
fig6, axes6 = plt.subplots(len(agent_ids), 1, figsize=(14, 9.2), sharex=True)
if len(agent_ids) == 1:
    axes6 = [axes6]
price_axes6 = []
for ax, agent_id in zip(axes6, agent_ids):
    one_agent = detail_agent.loc[detail_agent["agent_id"] == agent_id].sort_values("timestamp")
    profile = one_agent["agent_profile"].iloc[0] if "agent_profile" in one_agent.columns and not one_agent.empty else f"Agent {agent_id}"
    price_ax = ax.twinx()
    price_axes6.append(price_ax)
    ax.plot(one_agent["timestamp"], one_agent["e_bat"], color="#4C78A8", linewidth=1.1, label="Battery power")
    ax.plot(one_agent["timestamp"], one_agent["ev_charge_kw"], color="#E45756", linewidth=1.0, label="EV charging power")
    price_ax.plot(detail_step["timestamp"], detail_step["import_price"], color="black", linewidth=0.9, linestyle="--", alpha=0.75, label="Import price")
    ax.axhline(0.0, color="black", linewidth=0.8, alpha=0.5)
    ax.set_ylabel("kW")
    price_ax.set_ylabel("EUR/kWh")
    ax.set_title(f"{profile}: EV Charging and Battery Charging/Discharging")
    ax.grid(True, alpha=0.25)
power_lines, power_labels = axes6[0].get_legend_handles_labels()
price_lines, price_labels = price_axes6[0].get_legend_handles_labels()
axes6[0].legend(power_lines + price_lines, power_labels + price_labels, loc="upper right", frameon=False, ncol=3)
axes6[-1].set_xlabel("Time")
fig6.autofmt_xdate()
fig6.tight_layout()
figures.append((fig6, "ch6_3_soft_agent_ev_battery_price_madrl_base_progress_continuous_15days.png"))

fig7, ax7 = plt.subplots(figsize=(12, 5.4))
departure_soc_values = []
for record in records:
    spec = record["spec"]
    departure = record["agent"].loc[record["agent"]["step"] == EV_DEPARTURE_STEP].copy()
    if departure.empty:
        continue
    departure_daily = departure.groupby("episode_idx", as_index=False)["ev_soc"].min().sort_values("episode_idx")
    departure_daily["day"] = departure_daily["episode_idx"] + 1
    departure_soc_values.extend(departure_daily["ev_soc"].tolist())
    ax7.plot(
        departure_daily["day"],
        departure_daily["ev_soc"],
        marker="o",
        linewidth=1.7,
        markersize=4,
        label=spec["label"],
        color=spec["color"],
    )
ax7.axhline(EV_REQUIRED_SOC, color="black", linestyle="--", linewidth=1.2, label=f"Target SoC = {EV_REQUIRED_SOC:.2f}")
ax7.set_xlabel("Evaluation day")
ax7.set_ylabel("Minimum departure SoC")
ax7.set_xticks(range(1, 16))
y_min = np.floor((min(departure_soc_values) - 1e-9) / 0.05) * 0.05
ax7.set_ylim(max(0.0, y_min), 1.0)
ax7.grid(True, alpha=0.25)
ax7.legend(loc="lower right", frameon=False, ncol=2)
fig7.tight_layout()
figures.append((fig7, "ch6_3_soft_departure_soc_15days.png"))

# EV-constraint ablation: compare the minimum departure SoC of the non-projected Base variants.
selected_departure_specs = [
    {"key": "MADRL_BASE", "label": "Base Soft", "color": "#4C78A8"},
    {"key": "MADRL_BASE_COSTWEIGHT", "label": "Cost-Weighted Soft", "color": "#F58518"},
    {"key": "MADRL_BASE_PROGRESS_CONTINUOUS", "label": "Progress Soft", "color": "#54A24B"},
    {"key": "MADRL_BASE_EV_HARD_Continuous", "label": "Base Hard", "color": "#B279A2"},
]
fig8, ax8 = plt.subplots(figsize=(11.5, 5.2))
selected_departure_values = []
for spec in selected_departure_specs:
    agent_path = project_root / "artifacts" / "runs" / RUN_NAME / "results" / "lstm" / spec["key"] / "record" / "agent.parquet"
    agent = pd.read_parquet(agent_path)
    departure_daily = agent.loc[agent["step"] == EV_DEPARTURE_STEP].groupby("episode_idx", as_index=False)["ev_soc"].min().sort_values("episode_idx")
    departure_daily["day"] = departure_daily["episode_idx"] + 1
    selected_departure_values.extend(departure_daily["ev_soc"].tolist())
    ax8.plot(departure_daily["day"], departure_daily["ev_soc"], marker="o", linewidth=1.8, markersize=4.5, label=spec["label"], color=spec["color"])
ax8.axhline(EV_REQUIRED_SOC, color="black", linestyle="--", linewidth=1.3, label=f"Required SoC = {EV_REQUIRED_SOC:.2f}")
ax8.set_xlabel("Evaluation day")
ax8.set_ylabel("Minimum departure SoC")
ax8.set_xticks(range(1, 16))
ax8.set_ylim(max(0.0, np.floor((min(selected_departure_values) - 0.005) / 0.01) * 0.01), min(1.0, np.ceil((max(selected_departure_values) + 0.005) / 0.01) * 0.01))
ax8.grid(True, alpha=0.25)
ax8.legend(loc="lower right", frameon=False, ncol=2)
fig8.tight_layout()
figures.append((fig8, "ch6_3_base_variants_departure_soc.png"))

if savefigure == 1:
    THESIS_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    for fig, filename in figures:
        output_path = THESIS_FIGURE_DIR / filename
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Saved figure to: {output_path}")
    economic_table.to_csv(THESIS_FIGURE_DIR / "ch6_3_soft_economic_metrics.csv", index=False)
    ev_departure_table.to_csv(THESIS_FIGURE_DIR / "ch6_3_soft_ev_departure_soc.csv", index=False)
    grid_safety_table.to_csv(THESIS_FIGURE_DIR / "ch6_3_soft_grid_safety_metrics.csv", index=False)
    price_table.to_csv(THESIS_FIGURE_DIR / "ch6_3_soft_price_bin_charging.csv", index=False)
    print(f"Saved tables to: {THESIS_FIGURE_DIR}")
else:
    print("savefigure=0: figures and tables are displayed in the notebook output and were not saved.")

plt.show()


Economic metrics


,label,group,total_eur,system_other_cost_eur,storage_profit_total_eur,ev_charging_cost_eur
0,Base_soft,MADRL Soft,57.390,329.784,272.393,329.784
1,Base_Costweight_soft,MADRL Soft,251.639,531.580,102.748,354.387
2,Base_Progress_soft,MADRL Soft,188.694,335.677,146.983,335.677
3,Projection_Progress_soft,MADRL Soft,462.559,353.551,-109.008,353.551
4,Projection_EV_EME_soft,MADRL Soft,272.613,305.277,32.663,305.277
5,Base_EME_soft,MADRL Soft,94.284,195.215,100.931,195.215
6,ADMM-MPC Soft,Baseline,-24.855,210.693,235.548,210.693
7,Local MPC Soft,Baseline,-83.254,208.323,291.577,208.323
8,Rule-based with Battery,Baseline,274.009,386.895,112.886,386.895


EV departure SoC and EV feasibility metrics


,label,mean_departure_soc,min_departure_soc,departure_soc_below_req_count,departure_gap_total_soc,progress_gap_total_soc,emergency_added_kwh,projection_gap_kwh
0,Base_soft,0.9460,0.8782,2,0.0376,0.0000,0.0000,0.0
1,Base_Costweight_soft,0.9500,0.9500,0,0.0000,0.0000,0.0000,0.0
2,Base_Progress_soft,0.9500,0.9500,0,0.0000,14.5818,0.0000,0.0
3,Projection_Progress_soft,0.9463,0.8647,1,0.0353,7.1103,0.0000,0.0
4,Projection_EV_EME_soft,0.8702,0.5400,16,2.7520,0.0000,257.1622,0.0
5,Base_EME_soft,0.6112,0.4985,45,12.9948,0.0000,694.4813,0.0
6,ADMM-MPC Soft,0.8961,0.8940,45,0.1745,0.0000,0.0000,0.0
7,Local MPC Soft,0.8963,0.8940,45,0.1669,0.0000,0.0000,0.0
8,Rule-based with Battery,0.9500,0.9500,0,0.0000,0.0000,0.0000,0.0


Grid safety metrics


...

Saved figure to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures\ch6_3_soft_ev_charge_price_episode0.png


Saved figure to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures\ch6_3_soft_battery_power_price_episode0.png


Saved figure to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures\ch6_3_soft_agent_ev_battery_madrl_base_progress_continuous_episode0.png


Saved figure to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures\ch6_3_soft_departure_soc_15days.png
Saved tables to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures


C:\Users\20539\AppData\Local\Temp\ipykernel_21208\253807560.py:302: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# Chapter 6.4: MADRL hard-control postprocessing for the 7-day training run
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print


savefigure = 1  # 1: save figures and tables; 0: display only
THESIS_FIGURE_DIR = Path(r"D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures")
RUN_NAME = "madrl_traindays_7"
DT_HOURS = 0.25
EV_DEPARTURE_STEP = 28
EV_REQUIRED_SOC = 0.90
EV_COST_WEIGHT_FACTORS = {
    "MADRL_PROJECTION_EV_HARD_CostWeight": 1.5,
}

MADRL_HARD_SPECS = [
    {"key": "MADRL_BASE_EV_HARD_Continuous", "folder": "lstm", "label": "Base_hard", "stage": "Base_hard", "color": "#4C78A8"},
    {"key": "MADRL_PROJECTION_EV_HARD_CostWeight", "folder": "lstm", "label": "Projection_Costweight_hard", "stage": "Projection_Costweight_hard", "color": "#F58518"},
    {"key": "MADRL_PROJECTION_EV_HARD_CostWeight_PriceAware", "folder": "lstm", "label": "Projection_Priceaware_hard", "stage": "Projection_Priceaware_hard", "color": "#54A24B"},
    {"key": "MADRL_PROJECTION_EV_HARD_Continuous2", "folder": "lstm", "label": "Projection_hard", "stage": "Projection_hard", "color": "#B279A2"},
    {"key": "MADRL_PROJECTION_EV_HARD_Continuous3", "folder": "lstm", "label": "Projection_hard (SoC0.91)", "stage": "Projection_hard (SoC0.91)", "color": "#E45756"},
]

BASELINE_HARD_SPECS = [
    {"key": "admm_mpc_lstm_EV_hard", "folder": "lstm", "label": "ADMM-MPC Hard", "stage": "Baseline", "color": "#9D755D"},
    {"key": "local_mpc_lstm_EV_hard", "folder": "lstm", "label": "Local MPC Hard", "stage": "Baseline", "color": "#BAB0AC"},
    {"key": "rule_based_ev_with_battery", "folder": "perfect", "label": "Rule-based with Battery", "stage": "Baseline", "color": "#8CD17D"},
]

ALL_HARD_SPECS = MADRL_HARD_SPECS + BASELINE_HARD_SPECS

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "artifacts" / "runs").exists() and (path / "AAA-Thesis").exists():
            return path
    raise FileNotFoundError("Could not locate project root from the current notebook directory.")

def read_records(project_root: Path, spec: dict) -> dict:
    record_dir = project_root / "artifacts" / "runs" / RUN_NAME / "results" / spec["folder"] / spec["key"] / "record"
    required_files = ["metrics.parquet", "step.parquet", "agent.parquet"]
    missing = [name for name in required_files if not (record_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing {missing} under {record_dir}")
    return {"spec": spec, "metrics": pd.read_parquet(record_dir / "metrics.parquet"), "step": pd.read_parquet(record_dir / "step.parquet"), "agent": pd.read_parquet(record_dir / "agent.parquet")}

def series_sum(df: pd.DataFrame, col: str) -> float:
    return float(df[col].sum()) if col in df.columns else 0.0

def metric_value(metrics: pd.Series, col: str, default=np.nan) -> float:
    return float(metrics[col]) if col in metrics.index and pd.notna(metrics[col]) else default

def summarize_controller(record: dict) -> dict:
    spec = record["spec"]
    metrics = record["metrics"].iloc[0]
    step = record["step"]
    agent = record["agent"]
    departure = agent.loc[agent["step"] == EV_DEPARTURE_STEP].copy()
    ev_soc = departure["ev_soc"] if "ev_soc" in departure.columns else pd.Series(dtype=float)
    departure_gap = departure["ev_departure_gap"] if "ev_departure_gap" in departure.columns else pd.Series(dtype=float)
    raw_ev_charging_cost = series_sum(step, "ev_charging_cost_eur")
    ev_cost_weight = EV_COST_WEIGHT_FACTORS.get(spec["key"], 1.0)
    actual_ev_charging_cost = raw_ev_charging_cost / ev_cost_weight
    raw_total_eur = metric_value(metrics, "total_eur")
    actual_total_eur = raw_total_eur - raw_ev_charging_cost + actual_ev_charging_cost
    return {
        "label": spec["label"],
        "key": spec["key"],
        "group": "MADRL Hard" if spec in MADRL_HARD_SPECS else "Baseline",
        "stage": spec["stage"],
        "color": spec["color"],
        "total_eur": actual_total_eur,
        "raw_total_eur": raw_total_eur,
        "system_other_cost_eur": metric_value(metrics, "system_other_cost_eur"),
        "storage_profit_total_eur": metric_value(metrics, "storage_profit_total_eur"),
        "storage_charge_cost_total_eur": metric_value(metrics, "storage_charge_cost_total_eur"),
        "storage_discharge_revenue_total_eur": metric_value(metrics, "storage_discharge_revenue_total_eur"),
        "ev_charging_cost_eur": actual_ev_charging_cost,
        "raw_ev_charging_cost_eur": raw_ev_charging_cost,
        "ev_cost_weight_factor": ev_cost_weight,
        "mean_departure_soc": float(ev_soc.mean()) if len(ev_soc) else np.nan,
        "min_departure_soc": float(ev_soc.min()) if len(ev_soc) else np.nan,
        "departure_soc_below_req_count": int((ev_soc < EV_REQUIRED_SOC - 1e-9).sum()) if len(ev_soc) else 0,
        "departure_gap_total_soc": float(departure_gap.sum()) if len(departure_gap) else 0.0,
        "progress_gap_total_soc": series_sum(agent, "ev_progress_gap"),
        "price_aware_penalty_total": series_sum(agent, "ev_price_aware_penalty"),
        "emergency_added_kwh": series_sum(agent, "ev_emergency_added_kw") * DT_HOURS,
        "projection_gap_kwh": series_sum(agent, "ev_projection_gap_kw") * DT_HOURS,
        "voltage_violation_count": metric_value(metrics, "voltage_violation_count", 0.0),
        "voltage_violation_steps": metric_value(metrics, "voltage_violation_steps", 0.0),
        "min_vm_pu": metric_value(metrics, "min_vm_pu"),
        "max_vm_pu": metric_value(metrics, "max_vm_pu"),
        "trafo_overload_steps": metric_value(metrics, "trafo_overload_steps", 0.0),
        "trafo_loading_max_pct": metric_value(metrics, "trafo_loading_max_pct"),
        "feeder_netload_ramp_mean_abs_kw": metric_value(metrics, "feeder_netload_ramp_mean_abs_kw"),
        "battery_net_power_kw_mean_abs": metric_value(metrics, "battery_net_power_kw_mean_abs"),
    }

def build_price_bin_table(record: dict) -> dict:
    spec = record["spec"]
    step = record["step"][["episode_idx", "step", "import_price"]].copy()
    agent = record["agent"].merge(step, on=["episode_idx", "step"], how="left")
    agent = agent.loc[agent.get("ev_available", 1) == 1].copy()
    if agent.empty or "ev_charge_kw" not in agent.columns:
        return {"label": spec["label"], "weighted_charge_price_eur_per_kwh": np.nan, "total_ev_energy_kwh": 0.0, "low_price_energy_kwh": 0.0, "mid_price_energy_kwh": 0.0, "high_price_energy_kwh": 0.0, "low_price_share": 0.0, "high_price_share": 0.0}
    q_low, q_high = step["import_price"].quantile([1 / 3, 2 / 3])
    agent["ev_energy_kwh"] = agent["ev_charge_kw"].clip(lower=0.0) * DT_HOURS
    agent["price_bin"] = pd.cut(agent["import_price"], bins=[-np.inf, q_low, q_high, np.inf], labels=["low", "mid", "high"])
    energy_by_bin = agent.groupby("price_bin", observed=False)["ev_energy_kwh"].sum()
    total_energy = float(agent["ev_energy_kwh"].sum())
    weighted_price = float((agent["ev_energy_kwh"] * agent["import_price"]).sum() / total_energy) if total_energy > 0 else np.nan
    return {"label": spec["label"], "weighted_charge_price_eur_per_kwh": weighted_price, "total_ev_energy_kwh": total_energy, "low_price_energy_kwh": float(energy_by_bin.get("low", 0.0)), "mid_price_energy_kwh": float(energy_by_bin.get("mid", 0.0)), "high_price_energy_kwh": float(energy_by_bin.get("high", 0.0)), "low_price_share": float(energy_by_bin.get("low", 0.0) / total_energy) if total_energy > 0 else 0.0, "high_price_share": float(energy_by_bin.get("high", 0.0) / total_energy) if total_energy > 0 else 0.0}

project_root = find_project_root(Path.cwd().resolve())
hard_records = [read_records(project_root, spec) for spec in ALL_HARD_SPECS]
hard_summary_df = pd.DataFrame([summarize_controller(record) for record in hard_records])
hard_price_bin_df = pd.DataFrame([build_price_bin_table(record) for record in hard_records])

hard_economic_table = hard_summary_df[["label", "group", "total_eur", "system_other_cost_eur", "storage_profit_total_eur", "ev_charging_cost_eur", "ev_cost_weight_factor"]].round(3)
hard_ev_departure_table = hard_summary_df[["label", "mean_departure_soc", "min_departure_soc", "departure_soc_below_req_count", "departure_gap_total_soc", "emergency_added_kwh", "projection_gap_kwh"]].round(4)
hard_grid_safety_table = hard_summary_df[["label", "voltage_violation_count", "voltage_violation_steps", "min_vm_pu", "max_vm_pu", "trafo_overload_steps", "trafo_loading_max_pct"]].round(4)
hard_price_table = hard_price_bin_df.round(4)

print("Hard economic metrics")
display(hard_economic_table)
print("Hard EV departure SoC and feasibility metrics")
display(hard_ev_departure_table)
print("Hard grid safety metrics")
display(hard_grid_safety_table)
print("Hard EV charging and electricity-price relationship")
display(hard_price_table)

hard_madrl_summary = hard_summary_df.loc[hard_summary_df["group"] == "MADRL Hard"].copy()
hard_all_colors = hard_summary_df.set_index("label")["color"].to_dict()
hard_figures = []

fig1, ax1 = plt.subplots(figsize=(11.5, 4.8))
ax1.bar(hard_madrl_summary["stage"], hard_madrl_summary["total_eur"], color=hard_madrl_summary["color"])
ax1.axhline(0.0, color="black", linewidth=0.8)
ax1.set_title("MADRL Hard Logic Chain: Total Cost")
ax1.set_ylabel("Total cost (EUR)")
ax1.tick_params(axis="x", rotation=25)
ax1.grid(True, axis="y", alpha=0.25)
fig1.tight_layout()
hard_figures.append((fig1, "ch6_4_hard_logic_chain_total_cost.png"))

fig2, ax2 = plt.subplots(figsize=(12, 5.2))
ax2.bar(hard_summary_df["label"], hard_summary_df["total_eur"], color=[hard_all_colors[x] for x in hard_summary_df["label"]])
ax2.axhline(0.0, color="black", linewidth=0.8)
ax2.set_title("MADRL Hard Controllers and Baselines: Total Cost")
ax2.set_ylabel("Total cost (EUR)")
ax2.tick_params(axis="x", rotation=35)
ax2.grid(True, axis="y", alpha=0.25)
fig2.tight_layout()
hard_figures.append((fig2, "ch6_4_hard_controller_total_cost_comparison.png"))

fig3, axes3 = plt.subplots(1, 2, figsize=(13, 4.8))
axes3[0].bar(hard_summary_df["label"], hard_summary_df["voltage_violation_count"], color=[hard_all_colors[x] for x in hard_summary_df["label"]])
axes3[0].set_title("Voltage Violation Count")
axes3[0].set_ylabel("Count")
axes3[0].tick_params(axis="x", rotation=40)
axes3[0].grid(True, axis="y", alpha=0.25)
axes3[1].bar(hard_summary_df["label"], hard_summary_df["trafo_loading_max_pct"], color=[hard_all_colors[x] for x in hard_summary_df["label"]])
axes3[1].axhline(100.0, color="black", linewidth=0.9, linestyle="--")
axes3[1].set_title("Maximum Transformer Loading")
axes3[1].set_ylabel("Loading (%)")
axes3[1].tick_params(axis="x", rotation=40)
axes3[1].grid(True, axis="y", alpha=0.25)
fig3.tight_layout()
hard_figures.append((fig3, "ch6_4_hard_grid_safety_comparison.png"))

episode_to_plot = 0
fig4, ax4 = plt.subplots(figsize=(12, 5.4))
hard_departure_soc_values = []
for record in hard_records:
    spec = record["spec"]
    departure = record["agent"].loc[record["agent"]["step"] == EV_DEPARTURE_STEP].copy()
    if departure.empty:
        continue
    departure_daily = departure.groupby("episode_idx", as_index=False)["ev_soc"].min().sort_values("episode_idx")
    departure_daily["day"] = departure_daily["episode_idx"] + 1
    hard_departure_soc_values.extend(departure_daily["ev_soc"].tolist())
    ax4.plot(departure_daily["day"], departure_daily["ev_soc"], marker="o", linewidth=1.7, markersize=4, label=spec["label"], color=spec["color"])
ax4.axhline(EV_REQUIRED_SOC, color="black", linestyle="--", linewidth=1.2, label=f"Target SoC = {EV_REQUIRED_SOC:.2f}")
ax4.set_xlabel("Evaluation day")
ax4.set_ylabel("Minimum departure SoC")
ax4.set_xticks(range(1, 16))
hard_soc_ymin = max(0.0, np.floor((min(hard_departure_soc_values) - 0.005) / 0.01) * 0.01)
hard_soc_ymax = min(1.0, np.ceil((max(hard_departure_soc_values) + 0.005) / 0.01) * 0.01)
ax4.set_ylim(hard_soc_ymin, hard_soc_ymax)
ax4.grid(True, alpha=0.25)
ax4.legend(loc="center right", frameon=False, ncol=2)
fig4.tight_layout()
hard_figures.append((fig4, "ch6_4_hard_departure_soc_15days.png"))

hard_timestamp_tz = pd.to_datetime(hard_records[0]["agent"]["timestamp"]).dt.tz
hard_ev_window_start = pd.Timestamp("2020-04-02 18:00", tz=hard_timestamp_tz)
hard_ev_window_end = pd.Timestamp("2020-04-03 07:00", tz=hard_timestamp_tz)
fig5, ax5 = plt.subplots(figsize=(12, 5.4))
price_axis5 = ax5.twinx()
for record in hard_records[:len(MADRL_HARD_SPECS)]:
    spec = record["spec"]
    agent = record["agent"]
    agent_time = pd.to_datetime(agent["timestamp"])
    hard_ev_window_agent = agent.loc[agent_time.between(hard_ev_window_start, hard_ev_window_end, inclusive="both")].copy()
    ev_charge = hard_ev_window_agent.groupby("timestamp")["ev_charge_kw"].sum().sort_index()
    ax5.plot(ev_charge.index, ev_charge.values, label=spec["label"], color=spec["color"], linewidth=1.5)
hard_base_step_all = hard_records[0]["step"].copy()
hard_base_step_time = pd.to_datetime(hard_base_step_all["timestamp"])
hard_ev_window_step = hard_base_step_all.loc[hard_base_step_time.between(hard_ev_window_start, hard_ev_window_end, inclusive="both")].sort_values("timestamp")
price_axis5.plot(hard_ev_window_step["timestamp"], hard_ev_window_step["import_price"], color="black", linewidth=1.2, linestyle="--", label="Import price")
ax5.set_ylabel("Total EV charging power (kW)")
price_axis5.set_ylabel("Import price (EUR/kWh)")
ax5.set_xlabel("Time")
ax5.grid(True, alpha=0.25)
lines, labels = ax5.get_legend_handles_labels()
lines2, labels2 = price_axis5.get_legend_handles_labels()
ax5.legend(lines + lines2, labels + labels2, loc="upper left", frameon=False, ncol=2)
fig5.autofmt_xdate()
fig5.tight_layout()
hard_figures.append((fig5, "ch6_4_hard_ev_charge_price_episode0.png"))

base_step = hard_records[0]["step"].loc[hard_records[0]["step"]["episode_idx"] == episode_to_plot].sort_values("timestamp")
fig6, ax6 = plt.subplots(figsize=(12, 5.4))
price_axis6 = ax6.twinx()
for record in hard_records[:len(MADRL_HARD_SPECS)]:
    spec = record["spec"]
    day_agent = record["agent"].loc[record["agent"]["episode_idx"] == episode_to_plot].copy()
    battery_power = day_agent.groupby("timestamp")["e_bat"].sum().sort_index()
    ax6.plot(battery_power.index, battery_power.values, label=spec["label"], color=spec["color"], linewidth=1.5)
price_axis6.plot(base_step["timestamp"], base_step["import_price"], color="black", linewidth=1.2, linestyle="--", label="Import price")
ax6.axhline(0.0, color="black", linewidth=0.8, alpha=0.5)
ax6.set_ylabel("Total battery power (kW, + charge / - discharge)")
price_axis6.set_ylabel("Import price (EUR/kWh)")
ax6.set_xlabel("Time")
ax6.grid(True, alpha=0.25)
lines, labels = ax6.get_legend_handles_labels()
lines2, labels2 = price_axis6.get_legend_handles_labels()
ax6.legend(lines + lines2, labels + labels2, loc="upper left", frameon=False, ncol=2)
fig6.autofmt_xdate()
fig6.tight_layout()
hard_figures.append((fig6, "ch6_4_hard_battery_power_price_episode0.png"))

AGENT_DETAIL_CONTROLLER_KEY = "MADRL_PROJECTION_EV_HARD_Continuous3"
detail_record = next(record for record in hard_records if record["spec"]["key"] == AGENT_DETAIL_CONTROLLER_KEY)
detail_agent = detail_record["agent"].loc[detail_record["agent"]["episode_idx"] == episode_to_plot].copy()
agent_ids = sorted(detail_agent["agent_id"].unique())
fig7, axes7 = plt.subplots(len(agent_ids), 1, figsize=(12, 8.2), sharex=True)
if len(agent_ids) == 1:
    axes7 = [axes7]
for ax, agent_id in zip(axes7, agent_ids):
    one_agent = detail_agent.loc[detail_agent["agent_id"] == agent_id].sort_values("timestamp")
    profile = one_agent["agent_profile"].iloc[0] if "agent_profile" in one_agent.columns and not one_agent.empty else f"Agent {agent_id}"
    ax.plot(one_agent["timestamp"], one_agent["e_bat"], color="#4C78A8", linewidth=1.6, label="Battery power")
    ax.plot(one_agent["timestamp"], one_agent["ev_charge_kw"], color="#E45756", linewidth=1.4, label="EV charging power")
    ax.axhline(0.0, color="black", linewidth=0.8, alpha=0.5)
    ax.set_ylabel("kW")
    ax.set_title(f"{profile}: EV Charging and Battery Charging/Discharging")
    ax.grid(True, alpha=0.25)
axes7[0].legend(loc="upper right", frameon=False, ncol=2)
axes7[-1].set_xlabel("Time")
fig7.autofmt_xdate()
fig7.tight_layout()
hard_figures.append((fig7, "ch6_4_hard_agent_ev_battery_projection_hard_socmax091_episode0.png"))

# Grid-safety ablation: compare the four projected MADRL controllers used in Section 6.4.
projection_departure_specs = [
    {"key": "madrl_projection_safe_Continuous_Progress", "label": "Projected Progress Soft", "color": "#B279A2"},
    {"key": "MADRL_PROJECTION_EV_HARD_CostWeight", "label": "Cost-Weighted Projected Hard", "color": "#F58518"},
    {"key": "MADRL_PROJECTION_EV_HARD_CostWeight_PriceAware", "label": "Price-Aware Projected Hard", "color": "#54A24B"},
    {"key": "MADRL_PROJECTION_EV_HARD_Continuous2", "label": "Projected Hard", "color": "#4C78A8"},
]
fig8, ax8 = plt.subplots(figsize=(12, 5.4))
projection_departure_values = []
for spec in projection_departure_specs:
    agent_path = project_root / "artifacts" / "runs" / RUN_NAME / "results" / "lstm" / spec["key"] / "record" / "agent.parquet"
    agent = pd.read_parquet(agent_path)
    departure_daily = agent.loc[agent["step"] == EV_DEPARTURE_STEP].groupby("episode_idx", as_index=False)["ev_soc"].min().sort_values("episode_idx")
    departure_daily["day"] = departure_daily["episode_idx"] + 1
    projection_departure_values.extend(departure_daily["ev_soc"].tolist())
    ax8.plot(departure_daily["day"], departure_daily["ev_soc"], marker="o", linewidth=1.7, markersize=4, label=spec["label"], color=spec["color"])
ax8.axhline(EV_REQUIRED_SOC, color="black", linestyle="--", linewidth=1.3, label=f"Required SoC = {EV_REQUIRED_SOC:.2f}")
ax8.set_xlabel("Evaluation day")
ax8.set_ylabel("Minimum departure SoC")
ax8.set_xticks(range(1, 16))
ax8.set_ylim(max(0.0, np.floor((min(projection_departure_values) - 0.005) / 0.01) * 0.01), min(1.0, np.ceil((max(projection_departure_values) + 0.005) / 0.01) * 0.01))
ax8.grid(True, alpha=0.25)
ax8.legend(loc="lower right", frameon=False, ncol=2)
fig8.tight_layout()
hard_figures.append((fig8, "ch6_4_projection_variants_departure_soc.png"))

# Limitation figures with readable thesis labels.
limitation_groups = [
    ("ch6_6_eme_departure_soc.png", [("MADRL_BASE_PROGRESS_CONTINUOUS", "Progress Soft", "#54A24B"), ("MADRL_BASE_EV_EME", "Emergency Soft", "#E45756"), ("MADRL_PROJECTION_EV_EME", "Emergency Projected Soft", "#B279A2")]),
    ("ch6_6_soc091_departure_soc.png", [("MADRL_BASE_EV_HARD_Continuous", "Base Hard", "#F58518"), ("MADRL_PROJECTION_EV_HARD_Continuous2", "Projected Hard", "#4C78A8"), ("MADRL_PROJECTION_EV_HARD_Continuous3", "Projected Hard (EV SoC Upper Limit 0.91)", "#E45756")]),
]
for filename, specs in limitation_groups:
    fig_lim, ax_lim = plt.subplots(figsize=(12, 5.4))
    values = []
    for key, label, color in specs:
        agent_path = project_root / "artifacts" / "runs" / RUN_NAME / "results" / "lstm" / key / "record" / "agent.parquet"
        agent = pd.read_parquet(agent_path)
        daily = agent.loc[agent["step"] == EV_DEPARTURE_STEP].groupby("episode_idx", as_index=False)["ev_soc"].min().sort_values("episode_idx")
        values.extend(daily["ev_soc"].tolist())
        ax_lim.plot(daily["episode_idx"] + 1, daily["ev_soc"], marker="o", linewidth=1.7, markersize=4, label=label, color=color)
    ax_lim.axhline(EV_REQUIRED_SOC, color="black", linestyle="--", linewidth=1.3, label=f"Required SoC = {EV_REQUIRED_SOC:.2f}")
    ax_lim.set_xlabel("Evaluation day")
    ax_lim.set_ylabel("Minimum departure SoC")
    ax_lim.set_xticks(range(1, 16))
    ax_lim.set_ylim(max(0.0, np.floor((min(values) - 0.005) / 0.05) * 0.05), min(1.0, np.ceil((max(values) + 0.005) / 0.01) * 0.01))
    ax_lim.grid(True, alpha=0.25)
    ax_lim.legend(loc="lower right", frameon=False, ncol=2)
    fig_lim.tight_layout()
    hard_figures.append((fig_lim, filename))

if savefigure == 1:
    THESIS_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    for fig, filename in hard_figures:
        output_path = THESIS_FIGURE_DIR / filename
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Saved figure to: {output_path}")
    hard_economic_table.to_csv(THESIS_FIGURE_DIR / "ch6_4_hard_economic_metrics.csv", index=False)
    hard_ev_departure_table.to_csv(THESIS_FIGURE_DIR / "ch6_4_hard_ev_departure_soc.csv", index=False)
    hard_grid_safety_table.to_csv(THESIS_FIGURE_DIR / "ch6_4_hard_grid_safety_metrics.csv", index=False)
    hard_price_table.to_csv(THESIS_FIGURE_DIR / "ch6_4_hard_price_bin_charging.csv", index=False)
    print(f"Saved hard tables to: {THESIS_FIGURE_DIR}")
else:
    print("savefigure=0: hard figures and tables are displayed in the notebook output and were not saved.")

plt.show()


Hard economic metrics


,label,group,total_eur,system_other_cost_eur,storage_profit_total_eur,ev_charging_cost_eur,ev_cost_weight_factor
0,Base_hard,MADRL Hard,81.666,287.533,205.867,287.533,1.0
1,Projection_Costweight_hard,MADRL Hard,356.284,434.216,-66.807,289.477,1.5
2,Projection_Priceaware_hard,MADRL Hard,238.891,291.350,52.459,291.350,1.0
3,Projection_hard,MADRL Hard,186.037,373.424,187.387,373.424,1.0
4,Projection_hard (SoC0.91),MADRL Hard,494.247,288.425,-205.822,288.425,1.0
5,ADMM-MPC Hard,Baseline,-2.855,235.744,238.599,235.744,1.0
6,Local MPC Hard,Baseline,-36.374,235.681,272.055,235.681,1.0
7,Rule-based with Battery,Baseline,274.009,386.895,112.886,386.895,1.0


Hard EV departure SoC and feasibility metrics


,label,mean_departure_soc,min_departure_soc,departure_soc_below_req_count,departure_gap_total_soc,emergency_added_kwh,projection_gap_kwh
0,Base_hard,0.9098,0.90,1,0.0,0.0,9.7221
1,Projection_Costweight_hard,0.9084,0.90,6,0.0,0.0,71.2500
2,Projection_Priceaware_hard,0.9093,0.90,3,0.0,0.0,51.1275
3,Projection_hard,0.9500,0.95,0,0.0,0.0,0.0000
4,Projection_hard (SoC0.91),0.9093,0.90,3,0.0,0.0,13.4456
5,ADMM-MPC Hard,0.9000,0.90,45,0.0,0.0,1680.3110
6,Local MPC Hard,0.9000,0.90,45,0.0,0.0,1672.2630
7,Rule-based with Battery,0.9500,0.95,0,0.0,0.0,0.0000


Hard grid safety metrics


...

Saved figure to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures\ch6_4_hard_departure_soc_15days.png


Saved figure to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures\ch6_4_hard_ev_charge_price_episode0.png


Saved figure to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures\ch6_4_hard_battery_power_price_episode0.png


Saved figure to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures\ch6_4_hard_agent_ev_battery_projection_hard_socmax091_episode0.png
Saved hard tables to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures


C:\Users\20539\AppData\Local\Temp\ipykernel_21208\3160344584.py:278: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Chapter 6.5: Selected controller behavior comparison over the 15-day evaluation period
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print


savefigure = 1  # 1: save figures and tables; 0: display only
THESIS_FIGURE_DIR = Path(r"D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures")
RUN_NAME = "madrl_traindays_7"
DT_HOURS = 0.25

SELECTED_SPECS = [
    {"key": "MADRL_BASE", "folder": "lstm", "label": "Base_soft", "group": "Soft", "color": "#4C78A8"},
    {"key": "MADRL_BASE_PROGRESS_CONTINUOUS", "folder": "lstm", "label": "Base_Progress_soft", "group": "Soft", "color": "#54A24B"},
    {"key": "MADRL_BASE_EV_HARD_Continuous", "folder": "lstm", "label": "Base_hard", "group": "Hard", "color": "#F58518"},
    {"key": "MADRL_PROJECTION_EV_HARD_Continuous2", "folder": "lstm", "label": "Projection_hard", "group": "Hard", "color": "#B279A2"},
]

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "artifacts" / "runs").exists() and (path / "AAA-Thesis").exists():
            return path
    raise FileNotFoundError("Could not locate project root from the current notebook directory.")

def read_selected_records(project_root: Path, spec: dict) -> dict:
    record_dir = project_root / "artifacts" / "runs" / RUN_NAME / "results" / spec["folder"] / spec["key"] / "record"
    required_files = ["step.parquet", "agent.parquet"]
    missing = [name for name in required_files if not (record_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing {missing} under {record_dir}")
    return {"spec": spec, "step": pd.read_parquet(record_dir / "step.parquet"), "agent": pd.read_parquet(record_dir / "agent.parquet")}

def attach_price(record: dict) -> pd.DataFrame:
    step_price = record["step"][["episode_idx", "step", "timestamp", "import_price"]].copy()
    return record["agent"].merge(step_price, on=["episode_idx", "step", "timestamp"], how="left")

def summarize_average_prices(record: dict) -> dict:
    spec = record["spec"]
    agent = attach_price(record)
    ev_power = agent["ev_charge_kw"].clip(lower=0.0)
    ev_energy = ev_power * DT_HOURS
    battery_power = agent["e_bat"]
    battery_charge_power = battery_power.clip(lower=0.0)
    battery_discharge_power = -battery_power.clip(upper=0.0)
    battery_charge_energy = battery_charge_power * DT_HOURS
    battery_discharge_energy = battery_discharge_power * DT_HOURS
    ev_avg_price = (ev_energy * agent["import_price"]).sum() / ev_energy.sum() if ev_energy.sum() > 0 else np.nan
    battery_avg_charge_price = (battery_charge_energy * agent["import_price"]).sum() / battery_charge_energy.sum() if battery_charge_energy.sum() > 0 else np.nan
    battery_avg_discharge_price = (battery_discharge_energy * agent["import_price"]).sum() / battery_discharge_energy.sum() if battery_discharge_energy.sum() > 0 else np.nan
    return {
        "label": spec["label"],
        "group": spec["group"],
        "ev_avg_charge_price_eur_per_kwh": float(ev_avg_price),
        "ev_energy_kwh": float(ev_energy.sum()),
        "battery_avg_charge_price_eur_per_kwh": float(battery_avg_charge_price),
        "battery_charge_energy_kwh": float(battery_charge_energy.sum()),
        "battery_avg_discharge_price_eur_per_kwh": float(battery_avg_discharge_price),
        "battery_discharge_energy_kwh": float(battery_discharge_energy.sum()),
        "battery_price_spread_eur_per_kwh": float(battery_avg_discharge_price - battery_avg_charge_price),
    }

project_root = find_project_root(Path.cwd().resolve())
selected_records = [read_selected_records(project_root, spec) for spec in SELECTED_SPECS]
selected_price_table = pd.DataFrame([summarize_average_prices(record) for record in selected_records]).round(4)

print("Selected controllers: average EV and battery prices")
display(selected_price_table)

selected_figures = []

fig1, ax1 = plt.subplots(figsize=(14, 5.8))
price_axis1 = ax1.twinx()
for record in selected_records:
    spec = record["spec"]
    ev_charge = record["agent"].groupby("timestamp")["ev_charge_kw"].sum().sort_index()
    ax1.plot(ev_charge.index, ev_charge.values, label=spec["label"], color=spec["color"], linewidth=1.35)
base_step = selected_records[0]["step"].sort_values("timestamp")
price_axis1.plot(base_step["timestamp"], base_step["import_price"], color="black", linewidth=1.0, linestyle="--", label="Import price")
ax1.set_xlabel("Time")
ax1.set_ylabel("Total EV charging power (kW)")
price_axis1.set_ylabel("Import price (EUR/kWh)")
ax1.grid(True, alpha=0.25)
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = price_axis1.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc="upper left", frameon=False, ncol=3)
fig1.autofmt_xdate()
fig1.tight_layout()
selected_figures.append((fig1, "ch6_5_selected_ev_charge_price_15days.png"))

fig2, ax2 = plt.subplots(figsize=(14, 5.8))
price_axis2 = ax2.twinx()
for record in selected_records:
    spec = record["spec"]
    battery_power = record["agent"].groupby("timestamp")["e_bat"].sum().sort_index()
    ax2.plot(battery_power.index, battery_power.values, label=spec["label"], color=spec["color"], linewidth=1.35)
price_axis2.plot(base_step["timestamp"], base_step["import_price"], color="black", linewidth=1.0, linestyle="--", label="Import price")
ax2.axhline(0.0, color="black", linewidth=0.8, alpha=0.5)
ax2.set_xlabel("Time")
ax2.set_ylabel("Total battery power (kW, + charge / - discharge)")
price_axis2.set_ylabel("Import price (EUR/kWh)")
ax2.grid(True, alpha=0.25)
lines, labels = ax2.get_legend_handles_labels()
lines2, labels2 = price_axis2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc="upper left", frameon=False, ncol=3)
fig2.autofmt_xdate()
fig2.tight_layout()
selected_figures.append((fig2, "ch6_5_selected_battery_power_price_15days.png"))

if savefigure == 1:
    THESIS_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    for fig, filename in selected_figures:
        output_path = THESIS_FIGURE_DIR / filename
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
        print(f"Saved figure to: {output_path}")
    selected_price_table.to_csv(THESIS_FIGURE_DIR / "ch6_5_selected_average_prices.csv", index=False)
    print(f"Saved selected-controller table to: {THESIS_FIGURE_DIR}")
else:
    print("savefigure=0: selected-controller figures and table are displayed in the notebook output and were not saved.")

plt.show()


Selected controllers: average EV and battery prices


,label,group,ev_avg_charge_price_eur_per_kwh,ev_energy_kwh,battery_avg_charge_price_eur_per_kwh,battery_charge_energy_kwh,battery_avg_discharge_price_eur_per_kwh,battery_discharge_energy_kwh,battery_price_spread_eur_per_kwh
0,Base_soft,Soft,0.1714,1924.1712,0.1039,7209.2717,0.1589,6430.7510,0.0549
1,Base_Progress_soft,Soft,0.1723,1947.6669,0.1300,8537.5371,0.1634,7690.9807,0.0334
2,Base_hard,Hard,0.1600,1797.3481,0.1275,8712.3655,0.1710,7699.7998,0.0435
3,Projection_hard,Hard,0.1897,1968.3301,0.1295,8939.7873,0.1742,7720.7278,0.0447


Saved figure to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures\ch6_5_selected_ev_charge_price_15days.png


Saved figure to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures\ch6_5_selected_battery_power_price_15days.png
Saved selected-controller table to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures


C:\Users\20539\AppData\Local\Temp\ipykernel_21208\309058757.py:129: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
# Other plots for slides and presentations
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd


savefigure = 1  # 1: save figures; 0: display only
THESIS_FIGURE_DIR = Path(r"D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures")
RUN_NAME = "madrl_traindays_7"
CONTROLLER_KEY = "MADRL_BASE_COSTWEIGHT"
CONTROLLER_FOLDER = "lstm"
EPISODE_TO_PLOT = 0

def find_project_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "artifacts" / "runs").exists() and (path / "AAA-Thesis").exists():
            return path
    raise FileNotFoundError("Could not locate project root from the current notebook directory.")

project_root = find_project_root(Path.cwd().resolve())
record_dir = project_root / "artifacts" / "runs" / RUN_NAME / "results" / CONTROLLER_FOLDER / CONTROLLER_KEY / "record"
step_path = record_dir / "step.parquet"
agent_path = record_dir / "agent.parquet"
missing = [path for path in [step_path, agent_path] if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing required record files: {missing}")

step_df = pd.read_parquet(step_path).copy()
agent_df = pd.read_parquet(agent_path).copy()
day_step = step_df.loc[step_df["episode_idx"] == EPISODE_TO_PLOT].sort_values("timestamp").copy()
day_agent = agent_df.loc[agent_df["episode_idx"] == EPISODE_TO_PLOT].sort_values(["agent_id", "timestamp"]).copy()
if day_step.empty or day_agent.empty:
    raise ValueError(f"No records found for episode_idx={EPISODE_TO_PLOT} under {record_dir}")

agent_ids = sorted(day_agent["agent_id"].unique())
colors = ["#4C78A8", "#F58518", "#54A24B", "#B279A2", "#E45756", "#72B7B2"]
fig, axes = plt.subplots(2, 1, figsize=(12, 6.8), sharex=True)
price_axes = [ax.twinx() for ax in axes]

for agent_id, color in zip(agent_ids, colors):
    one_agent = day_agent.loc[day_agent["agent_id"] == agent_id]
    label = f"Agent {agent_id}"
    axes[0].plot(one_agent["timestamp"], one_agent["ev_charge_kw"], label=label, color=color, linewidth=1.6)
    axes[1].plot(one_agent["timestamp"], one_agent["e_bat"], label=label, color=color, linewidth=1.6)

for price_ax in price_axes:
    price_ax.plot(day_step["timestamp"], day_step["import_price"], color="black", linestyle="--", linewidth=1.1, label="Import price")
    price_ax.set_ylabel("Import price (EUR/kWh)")

axes[0].set_title("EV Soft Constraint + Cost Weight: EV Charging Power and Import Price")
axes[0].set_ylabel("EV charging power (kW)")
axes[0].grid(True, alpha=0.25)

axes[1].set_title("EV Soft Constraint + Cost Weight: Battery Power and Import Price")
axes[1].axhline(0.0, color="black", linewidth=0.8, alpha=0.55)
axes[1].set_ylabel("Battery power (kW, + charge / - discharge)")
axes[1].set_xlabel("Time")
axes[1].grid(True, alpha=0.25)

for ax, price_ax in zip(axes, price_axes):
    lines, labels = ax.get_legend_handles_labels()
    price_lines, price_labels = price_ax.get_legend_handles_labels()
    ax.legend(lines + price_lines, labels + price_labels, loc="upper left", frameon=False, ncol=4)

fig.autofmt_xdate()
fig.tight_layout()

if savefigure == 1:
    THESIS_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    output_path = THESIS_FIGURE_DIR / "ch6_3_soft_costweight_ev_battery_power_price_episode0.png"
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"Saved figure to: {output_path}")
else:
    print("savefigure=0: figure is displayed in the notebook output and was not saved.")

plt.show()

# One-agent illustration: EV charging and battery charging compete for the same low-price periods.
full_agent = agent_df.merge(step_df[["episode_idx", "step", "timestamp", "import_price"]], on=["episode_idx", "step", "timestamp"], how="left")
agent_scores = []
for agent_id, one_agent in full_agent.groupby("agent_id"):
    ev_energy = one_agent["ev_charge_kw"].clip(lower=0.0) * 0.25
    battery_charge_energy = one_agent["e_bat"].clip(lower=0.0) * 0.25
    battery_discharge_energy = -one_agent["e_bat"].clip(upper=0.0) * 0.25
    ev_avg_price = (ev_energy * one_agent["import_price"]).sum() / ev_energy.sum() if ev_energy.sum() > 0 else float("nan")
    battery_charge_price = (battery_charge_energy * one_agent["import_price"]).sum() / battery_charge_energy.sum() if battery_charge_energy.sum() > 0 else float("nan")
    battery_discharge_price = (battery_discharge_energy * one_agent["import_price"]).sum() / battery_discharge_energy.sum() if battery_discharge_energy.sum() > 0 else float("nan")
    agent_scores.append({"agent_id": agent_id, "ev_avg_price": ev_avg_price, "battery_spread": battery_discharge_price - battery_charge_price})
agent_score_df = pd.DataFrame(agent_scores)
worst_agent_id = int(agent_score_df.sort_values(["ev_avg_price", "battery_spread"], ascending=[False, True]).iloc[0]["agent_id"])

worst_day = day_agent.loc[day_agent["agent_id"] == worst_agent_id].merge(day_step[["episode_idx", "step", "timestamp", "import_price"]], on=["episode_idx", "step", "timestamp"], how="left")
low_price_threshold = day_step["import_price"].quantile(0.25)
time_values = worst_day["timestamp"]
ev_power = worst_day["ev_charge_kw"].clip(lower=0.0)
battery_charge_power = worst_day["e_bat"].clip(lower=0.0)
battery_discharge_power = -worst_day["e_bat"].clip(upper=0.0)

fig2, ax = plt.subplots(figsize=(12, 4.8))
price_ax = ax.twinx()
low_price = day_step["import_price"] <= low_price_threshold
for idx in day_step.index[low_price]:
    start = day_step.loc[idx, "timestamp"]
    next_rows = day_step.loc[day_step["timestamp"] > start, "timestamp"]
    end = next_rows.iloc[0] if len(next_rows) else start + pd.Timedelta(minutes=15)
    ax.axvspan(start, end, color="#D7E9FF", alpha=0.45, linewidth=0)

ax.plot(time_values, ev_power, color="#4C78A8", linewidth=1.8, label="EV charging power")
ax.fill_between(time_values, ev_power, color="#4C78A8", alpha=0.18)
ax.plot(time_values, battery_charge_power, color="#F58518", linewidth=1.8, label="Battery charging power")
ax.fill_between(time_values, battery_charge_power, color="#F58518", alpha=0.18)
ax.plot(time_values, battery_discharge_power, color="#54A24B", linewidth=1.35, linestyle=":", label="Battery discharging power")
price_ax.plot(day_step["timestamp"], day_step["import_price"], color="black", linestyle="--", linewidth=1.15, label="Import price")
price_ax.axhline(low_price_threshold, color="black", linewidth=0.8, alpha=0.35, linestyle=":", label="Low-price threshold")

ax.set_title(f"EV Soft Constraint + Cost Weight: EV and Battery Compete for Low-Price Periods (Agent {worst_agent_id})")
ax.set_ylabel("Power (kW)")
ax.set_xlabel("Time")
price_ax.set_ylabel("Import price (EUR/kWh)")
ax.grid(True, alpha=0.25)
lines, labels = ax.get_legend_handles_labels()
price_lines, price_labels = price_ax.get_legend_handles_labels()
ax.legend(lines + price_lines, labels + price_labels, loc="upper left", frameon=False, ncol=3)
fig2.autofmt_xdate()
fig2.tight_layout()

if savefigure == 1:
    output_path = THESIS_FIGURE_DIR / "ch6_3_soft_costweight_agent0_ev_battery_low_price_competition_episode0.png"
    fig2.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"Saved figure to: {output_path}")
else:
    print("savefigure=0: competition figure is displayed in the notebook output and was not saved.")

plt.show()

# Soft EV comparison: progress penalty encourages gradual charging instead of concentrated charging.
SOFT_EV_PROFILE_SPECS = [
    {"key": "MADRL_BASE_COSTWEIGHT", "label": "Base_Costweight_soft", "color": "#F58518"},
    {"key": "MADRL_BASE_PROGRESS_CONTINUOUS", "label": "Base_Progress_soft", "color": "#54A24B"},
]

ev_profile_records = []
base_profile_step = pd.read_parquet(project_root / "artifacts" / "runs" / RUN_NAME / "results" / "lstm" / SOFT_EV_PROFILE_SPECS[0]["key"] / "record" / "step.parquet")
episode_day_start = base_profile_step.loc[base_profile_step["episode_idx"] == EPISODE_TO_PLOT, "timestamp"].min().normalize()
ev_window_start = episode_day_start + pd.Timedelta(hours=18)
ev_window_end = ev_window_start + pd.Timedelta(hours=13)
for spec in SOFT_EV_PROFILE_SPECS:
    spec_record_dir = project_root / "artifacts" / "runs" / RUN_NAME / "results" / "lstm" / spec["key"] / "record"
    spec_step = pd.read_parquet(spec_record_dir / "step.parquet")
    spec_agent = pd.read_parquet(spec_record_dir / "agent.parquet")
    spec_window_agent = spec_agent.loc[(spec_agent["timestamp"] >= ev_window_start) & (spec_agent["timestamp"] <= ev_window_end)].copy()
    ev_power = spec_window_agent.groupby("timestamp")["ev_charge_kw"].sum().sort_index()
    ev_energy_cum = (ev_power * 0.25).cumsum()
    ev_profile_records.append({"spec": spec, "step": spec_step, "ev_power": ev_power, "ev_energy_cum": ev_energy_cum})

price_step = ev_profile_records[0]["step"].loc[(ev_profile_records[0]["step"]["timestamp"] >= ev_window_start) & (ev_profile_records[0]["step"]["timestamp"] <= ev_window_end)].sort_values("timestamp").copy()
low_price_threshold = price_step["import_price"].quantile(0.25)
fig3, axes3 = plt.subplots(2, 1, figsize=(12, 6.4), sharex=True, gridspec_kw={"height_ratios": [1.45, 1.0]})
price_ax3 = axes3[0].twinx()

low_price_mask = price_step["import_price"] <= low_price_threshold
for idx in price_step.index[low_price_mask]:
    start = price_step.loc[idx, "timestamp"]
    next_rows = price_step.loc[price_step["timestamp"] > start, "timestamp"]
    end = next_rows.iloc[0] if len(next_rows) else start + pd.Timedelta(minutes=15)
    axes3[0].axvspan(start, end, color="#D7E9FF", alpha=0.42, linewidth=0)
    axes3[1].axvspan(start, end, color="#D7E9FF", alpha=0.28, linewidth=0)

for record in ev_profile_records:
    spec = record["spec"]
    axes3[0].plot(record["ev_power"].index, record["ev_power"].values, color=spec["color"], linewidth=1.7, label=spec["label"])
    axes3[1].plot(record["ev_energy_cum"].index, record["ev_energy_cum"].values, color=spec["color"], linewidth=1.8, label=spec["label"])

price_ax3.plot(price_step["timestamp"], price_step["import_price"], color="black", linestyle="--", linewidth=1.15, label="Import price")
price_ax3.axhline(low_price_threshold, color="black", linewidth=0.8, alpha=0.35, linestyle=":", label="Low-price threshold")

axes3[0].set_ylabel("Total EV charging power (kW)")
price_ax3.set_ylabel("Import price (EUR/kWh)")
axes3[0].grid(True, alpha=0.25)
axes3[1].set_ylabel("Cumulative EV energy (kWh)")
axes3[1].set_xlabel("Time")
axes3[1].grid(True, alpha=0.25)

lines, labels = axes3[0].get_legend_handles_labels()
price_lines, price_labels = price_ax3.get_legend_handles_labels()
axes3[0].legend(lines + price_lines, labels + price_labels, loc="upper left", frameon=False, ncol=3)
axes3[1].legend(loc="upper left", frameon=False, ncol=3)
fig3.autofmt_xdate()
fig3.tight_layout()

if savefigure == 1:
    output_path = THESIS_FIGURE_DIR / "ppt_soft_ev_charging_base_progress_vs_costweight_large_fonts.png"
    fig3.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"Saved figure to: {output_path}")
else:
    print("savefigure=0: progress EV charging comparison figure is displayed and was not saved.")

plt.show()

# Soft Projection_Progress_soft: grid safety and unprofitable battery behavior.
PROJECTION_PROGRESS_KEY = "madrl_projection_safe_Continuous_Progress"
projection_record_dir = project_root / "artifacts" / "runs" / RUN_NAME / "results" / "lstm" / PROJECTION_PROGRESS_KEY / "record"
projection_step = pd.read_parquet(projection_record_dir / "step.parquet")
projection_agent = pd.read_parquet(projection_record_dir / "agent.parquet")
projection_day_step = projection_step.loc[projection_step["episode_idx"] == EPISODE_TO_PLOT].sort_values("timestamp").copy()
projection_day_agent = projection_agent.loc[projection_agent["episode_idx"] == EPISODE_TO_PLOT].sort_values(["agent_id", "timestamp"]).copy()
battery_power = projection_day_agent.groupby("timestamp")["e_bat"].sum().sort_index()
episode_storage_profit = projection_day_step["storage_profit_eur"].sum() if "storage_profit_eur" in projection_day_step.columns else float("nan")

fig4, axes4 = plt.subplots(2, 1, figsize=(12, 6.6), sharex=True, gridspec_kw={"height_ratios": [1.25, 1.0]})
trafo_ax4 = axes4[0].twinx()
price_ax4 = axes4[1].twinx()

axes4[0].plot(projection_day_step["timestamp"], projection_day_step["root_net_exchange_kw"], color="#4C78A8", linewidth=1.55, label="Root net exchange")
axes4[0].plot(projection_day_step["timestamp"], projection_day_step["feeder_post_action_net_load_kw"], color="#72B7B2", linewidth=1.35, linestyle="-.", label="Post-action feeder load")
trafo_ax4.plot(projection_day_step["timestamp"], projection_day_step["trafo_loading_pct_max"], color="#E45756", linewidth=1.45, label="Max transformer loading")
trafo_ax4.axhline(100.0, color="#E45756", linewidth=0.9, linestyle=":", alpha=0.85, label="Transformer limit")
axes4[0].axhline(0.0, color="black", linewidth=0.75, alpha=0.5)
axes4[0].set_title("Soft Projection_Progress_soft: Grid Safety after Action Projection")
axes4[0].set_ylabel("Power (kW)")
trafo_ax4.set_ylabel("Transformer loading (%)")
axes4[0].grid(True, alpha=0.25)
lines, labels = axes4[0].get_legend_handles_labels()
trafo_lines, trafo_labels = trafo_ax4.get_legend_handles_labels()
axes4[0].legend(lines + trafo_lines, labels + trafo_labels, loc="upper left", frameon=False, ncol=2)

axes4[1].plot(battery_power.index, battery_power.values, color="#F58518", linewidth=1.6, label="Total battery power")
axes4[1].fill_between(battery_power.index, battery_power.values, 0, where=battery_power.values >= 0, color="#F58518", alpha=0.18, interpolate=True, label="Charge")
axes4[1].fill_between(battery_power.index, battery_power.values, 0, where=battery_power.values < 0, color="#54A24B", alpha=0.18, interpolate=True, label="Discharge")
price_ax4.plot(projection_day_step["timestamp"], projection_day_step["import_price"], color="black", linestyle="--", linewidth=1.1, label="Import price")
axes4[1].axhline(0.0, color="black", linewidth=0.8, alpha=0.55)
axes4[1].set_title(f"Battery Power vs Price: episode storage profit = {episode_storage_profit:.2f} EUR")
axes4[1].set_ylabel("Battery power (kW, + charge / - discharge)")
axes4[1].set_xlabel("Time")
price_ax4.set_ylabel("Import price (EUR/kWh)")
axes4[1].grid(True, alpha=0.25)
lines, labels = axes4[1].get_legend_handles_labels()
price_lines, price_labels = price_ax4.get_legend_handles_labels()
axes4[1].legend(lines + price_lines, labels + price_labels, loc="upper left", frameon=False, ncol=3)
fig4.autofmt_xdate()
fig4.tight_layout()

if savefigure == 1:
    output_path = THESIS_FIGURE_DIR / "ch6_3_soft_projection_progress_grid_safety_battery_price_episode0.png"
    fig4.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"Saved figure to: {output_path}")
else:
    print("savefigure=0: projection progress safety and battery figure is displayed and was not saved.")

plt.show()

# Base_hard: EV charging, battery operation, price response, and grid safety.
BASE_HARD_KEY = "MADRL_BASE_EV_HARD_Continuous"
base_hard_record_dir = project_root / "artifacts" / "runs" / RUN_NAME / "results" / "lstm" / BASE_HARD_KEY / "record"
base_hard_step = pd.read_parquet(base_hard_record_dir / "step.parquet")
base_hard_agent = pd.read_parquet(base_hard_record_dir / "agent.parquet")
base_hard_day_step = base_hard_step.loc[base_hard_step["episode_idx"] == EPISODE_TO_PLOT].sort_values("timestamp").copy()
base_hard_day_agent = base_hard_agent.loc[base_hard_agent["episode_idx"] == EPISODE_TO_PLOT].sort_values(["agent_id", "timestamp"]).copy()
base_hard_battery_power = base_hard_day_agent.groupby("timestamp")["e_bat"].sum().sort_index()
base_hard_ev_power = base_hard_day_agent.groupby("timestamp")["ev_charge_kw"].sum().sort_index()
base_hard_storage_profit = base_hard_day_step["storage_profit_eur"].sum() if "storage_profit_eur" in base_hard_day_step.columns else float("nan")

fig5, axes5 = plt.subplots(2, 1, figsize=(12, 6.6), sharex=True)
price_ax5_top = axes5[0].twinx()
price_ax5_bottom = axes5[1].twinx()
axes5[0].plot(base_hard_ev_power.index, base_hard_ev_power.values, color="#4C78A8", linewidth=1.65, label="Total EV charging power")
axes5[0].fill_between(base_hard_ev_power.index, base_hard_ev_power.values, color="#4C78A8", alpha=0.18)
price_ax5_top.plot(base_hard_day_step["timestamp"], base_hard_day_step["import_price"], color="black", linestyle="--", linewidth=1.1, label="Import price")
axes5[0].set_title("Base_hard: EV Charging Power and Import Price")
axes5[0].set_ylabel("EV charging power (kW)")
price_ax5_top.set_ylabel("Import price (EUR/kWh)")
axes5[0].grid(True, alpha=0.25)
lines, labels = axes5[0].get_legend_handles_labels()
price_lines, price_labels = price_ax5_top.get_legend_handles_labels()
axes5[0].legend(lines + price_lines, labels + price_labels, loc="upper left", frameon=False, ncol=2)

axes5[1].plot(base_hard_battery_power.index, base_hard_battery_power.values, color="#F58518", linewidth=1.65, label="Total battery power")
axes5[1].fill_between(base_hard_battery_power.index, base_hard_battery_power.values, 0, where=base_hard_battery_power.values >= 0, color="#F58518", alpha=0.18, interpolate=True, label="Charge")
axes5[1].fill_between(base_hard_battery_power.index, base_hard_battery_power.values, 0, where=base_hard_battery_power.values < 0, color="#54A24B", alpha=0.18, interpolate=True, label="Discharge")
price_ax5_bottom.plot(base_hard_day_step["timestamp"], base_hard_day_step["import_price"], color="black", linestyle="--", linewidth=1.1, label="Import price")
axes5[1].axhline(0.0, color="black", linewidth=0.8, alpha=0.55)
axes5[1].set_title(f"Base_hard: Battery Power and Import Price, storage profit = {base_hard_storage_profit:.2f} EUR")
axes5[1].set_ylabel("Battery power (kW, + charge / - discharge)")
axes5[1].set_xlabel("Time")
price_ax5_bottom.set_ylabel("Import price (EUR/kWh)")
axes5[1].grid(True, alpha=0.25)
lines, labels = axes5[1].get_legend_handles_labels()
price_lines, price_labels = price_ax5_bottom.get_legend_handles_labels()
axes5[1].legend(lines + price_lines, labels + price_labels, loc="upper left", frameon=False, ncol=3)
fig5.autofmt_xdate()
fig5.tight_layout()

if savefigure == 1:
    output_path = THESIS_FIGURE_DIR / "ch6_4_hard_base_ev_battery_power_price_episode0.png"
    fig5.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"Saved figure to: {output_path}")
else:
    print("savefigure=0: hard base EV and battery figure is displayed and was not saved.")

plt.show()

fig6, axes6 = plt.subplots(2, 1, figsize=(12, 6.6), sharex=True)
trafo_ax6 = axes6[0].twinx()
axes6[0].plot(base_hard_day_step["timestamp"], base_hard_day_step["root_net_exchange_kw"], color="#4C78A8", linewidth=1.55, label="Root net exchange")
axes6[0].plot(base_hard_day_step["timestamp"], base_hard_day_step["feeder_post_action_net_load_kw"], color="#72B7B2", linewidth=1.35, linestyle="-.", label="Post-action feeder load")
trafo_ax6.plot(base_hard_day_step["timestamp"], base_hard_day_step["trafo_loading_pct_max"], color="#E45756", linewidth=1.45, label="Max transformer loading")
trafo_ax6.axhline(100.0, color="#E45756", linewidth=0.9, linestyle=":", alpha=0.85, label="Transformer limit")
axes6[0].axhline(0.0, color="black", linewidth=0.75, alpha=0.5)
axes6[0].set_title("Base_hard: Grid Power and Transformer Loading")
axes6[0].set_ylabel("Power (kW)")
trafo_ax6.set_ylabel("Transformer loading (%)")
axes6[0].grid(True, alpha=0.25)
lines, labels = axes6[0].get_legend_handles_labels()
trafo_lines, trafo_labels = trafo_ax6.get_legend_handles_labels()
axes6[0].legend(lines + trafo_lines, labels + trafo_labels, loc="upper left", frameon=False, ncol=2)

axes6[1].plot(base_hard_day_step["timestamp"], base_hard_day_step["min_vm_pu"], color="#4C78A8", linewidth=1.45, label="Min voltage")
axes6[1].plot(base_hard_day_step["timestamp"], base_hard_day_step["max_vm_pu"], color="#F58518", linewidth=1.45, label="Max voltage")
axes6[1].axhline(0.95, color="#4C78A8", linewidth=0.9, linestyle=":", alpha=0.85, label="Voltage lower limit")
axes6[1].axhline(1.05, color="#F58518", linewidth=0.9, linestyle=":", alpha=0.85, label="Voltage upper limit")
axes6[1].set_title("Base_hard: Voltage Range")
axes6[1].set_ylabel("Voltage (p.u.)")
axes6[1].set_xlabel("Time")
axes6[1].grid(True, alpha=0.25)
axes6[1].legend(loc="upper left", frameon=False, ncol=2)
fig6.autofmt_xdate()
fig6.tight_layout()

if savefigure == 1:
    output_path = THESIS_FIGURE_DIR / "ch6_4_hard_base_grid_safety_episode0.png"
    fig6.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"Saved figure to: {output_path}")
else:
    print("savefigure=0: hard base grid safety figure is displayed and was not saved.")

plt.show()

# Projection_hard: per-agent EV charging, battery power, and import price.
PROJECTION_HARD_KEY = "MADRL_PROJECTION_EV_HARD_Continuous2"
projection_hard_record_dir = project_root / "artifacts" / "runs" / RUN_NAME / "results" / "lstm" / PROJECTION_HARD_KEY / "record"
projection_hard_step = pd.read_parquet(projection_hard_record_dir / "step.parquet")
projection_hard_agent = pd.read_parquet(projection_hard_record_dir / "agent.parquet")
projection_hard_plot_step = projection_hard_step.sort_values(["episode_idx", "step", "timestamp"]).copy()
projection_hard_plot_agent = projection_hard_agent.sort_values(["agent_id", "episode_idx", "step", "timestamp"]).copy()
agent_ids = sorted(projection_hard_plot_agent["agent_id"].unique())
fig7, axes7 = plt.subplots(len(agent_ids), 1, figsize=(12, 7.4), sharex=True)
if len(agent_ids) == 1:
    axes7 = [axes7]

for ax, agent_id in zip(axes7, agent_ids):
    one_agent = projection_hard_plot_agent.loc[projection_hard_plot_agent["agent_id"] == agent_id].sort_values(["episode_idx", "step", "timestamp"])
    price_ax = ax.twinx()
    profile = one_agent["agent_profile"].iloc[0] if "agent_profile" in one_agent.columns and not one_agent.empty else f"Agent {agent_id}"
    ax.plot(one_agent["timestamp"], one_agent["ev_charge_kw"], color="#4C78A8", linewidth=1.55, label="EV charging power")
    ax.plot(one_agent["timestamp"], one_agent["e_bat"], color="#F58518", linewidth=1.45, label="Battery power")
    ax.axhline(0.0, color="black", linewidth=0.75, alpha=0.5)
    price_ax.plot(projection_hard_plot_step["timestamp"], projection_hard_plot_step["import_price"], color="black", linestyle="--", linewidth=1.0, label="Import price")
    ax.set_ylabel("Power (kW)")
    price_ax.set_ylabel("Price")
    ax.set_title(f"Projection_hard: {profile}")
    ax.grid(True, alpha=0.25)
    lines, labels = ax.get_legend_handles_labels()
    price_lines, price_labels = price_ax.get_legend_handles_labels()
    ax.legend(lines + price_lines, labels + price_labels, loc="upper left", frameon=False, ncol=3)

axes7[-1].set_xlabel("Time")
fig7.autofmt_xdate()
fig7.tight_layout()

if savefigure == 1:
    output_path = THESIS_FIGURE_DIR / "ch6_4_projection_hard_agent_ev_battery_power_price_15days.png"
    fig7.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"Saved figure to: {output_path}")
else:
    print("savefigure=0: projection hard per-agent EV and battery figure is displayed and was not saved.")

plt.show()


Saved figure to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures\ch6_3_soft_costweight_ev_battery_power_price_episode0.png


C:\Users\20539\AppData\Local\Temp\ipykernel_21208\237063869.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved figure to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures\ch6_3_soft_costweight_agent0_ev_battery_low_price_competition_episode0.png


C:\Users\20539\AppData\Local\Temp\ipykernel_21208\237063869.py:137: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\20539\AppData\Local\Temp\ipykernel_21208\237063869.py:161: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig3, axes3 = plt.subplots(2, 1, figsize=(12, 6.4), sharex=True, gridspec_kw={"height_ratios": [1.45, 1.0]})


Saved figure to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures\ch6_3_soft_progress_gradual_ev_charging_vs_base_costweight_episode0.png


...

C:\Users\20539\AppData\Local\Temp\ipykernel_21208\237063869.py:304: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved figure to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures\ch6_4_hard_base_grid_safety_episode0.png


C:\Users\20539\AppData\Local\Temp\ipykernel_21208\237063869.py:340: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Saved figure to: D:\RL_Thesis_Project\MADRL_ESS\AAA-Thesis\Thesis\figures\ch6_4_projection_hard_agent_ev_battery_power_price_15days.png


C:\Users\20539\AppData\Local\Temp\ipykernel_21208\237063869.py:381: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Projection_hard: clear per-agent EV, signed battery operation, and import price over 15 days.
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd


savefigure = 1
PROJECT_ROOT = Path(r"D:\RL_Thesis_Project\MADRL_ESS")
PPT_FIGURE_DIR = PROJECT_ROOT / "PPT_Equations" / "PPT_figures"
PPT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
THESIS_FIGURE_DIR = PROJECT_ROOT / "AAA-Thesis" / "Thesis" / "figures"
THESIS_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RUN_NAME = "madrl_traindays_7"
PROJECTION_HARD_KEY = "MADRL_PROJECTION_EV_HARD_Continuous2"
CONTROLLER_FOLDER = "lstm"

record_dir = PROJECT_ROOT / "artifacts" / "runs" / RUN_NAME / "results" / CONTROLLER_FOLDER / PROJECTION_HARD_KEY / "record"
step_df = pd.read_parquet(record_dir / "step.parquet").copy()
agent_df = pd.read_parquet(record_dir / "agent.parquet").copy()
step_df["timestamp"] = pd.to_datetime(step_df["timestamp"].astype(str).str.replace(r"[+-]\d{2}:\d{2}$", "", regex=True))
agent_df["timestamp"] = pd.to_datetime(agent_df["timestamp"].astype(str).str.replace(r"[+-]\d{2}:\d{2}$", "", regex=True))

plot_step = step_df.sort_values(["episode_idx", "step", "timestamp"]).copy()
plot_agent = agent_df.sort_values(["agent_id", "episode_idx", "step", "timestamp"]).copy()
if plot_step.empty or plot_agent.empty:
    raise ValueError(f"No records found under {record_dir}")

plt.rcParams.update({
    "font.size": 17,
    "axes.titlesize": 19,
    "axes.labelsize": 17,
    "legend.fontsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "axes.linewidth": 1.1,
})

agent_ids = sorted(plot_agent["agent_id"].unique())
fig, axes = plt.subplots(len(agent_ids), 1, figsize=(18, 10.2), sharex=True, constrained_layout=True)
if len(agent_ids) == 1:
    axes = [axes]

for ax, agent_id in zip(axes, agent_ids):
    one_agent = plot_agent.loc[plot_agent["agent_id"] == agent_id].sort_values(["episode_idx", "step", "timestamp"]).copy()
    profile = one_agent["agent_profile"].iloc[0] if "agent_profile" in one_agent.columns else f"Agent {agent_id}"
    price_ax = ax.twinx()
    t = one_agent["timestamp"]
    ev_power = one_agent["ev_charge_kw"].clip(lower=0.0)
    bat_power = one_agent["e_bat"]

    ax.plot(t, ev_power, color="#4C78A8", linewidth=2.05, drawstyle="steps-post", label="EV charging")
    ax.plot(t, bat_power, color="#F58518", linewidth=1.45, alpha=0.88, label="Battery power (+ charge / - discharge)")
    price_ax.plot(plot_step["timestamp"], plot_step["import_price"], color="#111111", linestyle="--", linewidth=1.30, alpha=0.88, label="Import price")

    ax.axhline(0.0, color="#333333", linewidth=0.9, alpha=0.7)
    ax.set_ylim(-55, 55)
    price_ax.set_ylim(0, max(0.32, plot_step["import_price"].max() * 1.05))
    ax.set_ylabel("Power (kW)")
    price_ax.set_ylabel("Price")
    ax.set_title(f"Projection_hard: {profile}")
    ax.grid(True, axis="y", alpha=0.28)
    ax.grid(True, axis="x", alpha=0.14)

    handles, labels = ax.get_legend_handles_labels()
    price_handles, price_labels = price_ax.get_legend_handles_labels()
    ax.legend(handles + price_handles, labels + price_labels, loc="upper left", frameon=False, ncol=3)

axes[-1].set_xlabel("Time")
axes[-1].xaxis.set_major_locator(mdates.DayLocator(interval=2))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
for label in axes[-1].get_xticklabels():
    label.set_rotation(25)
    label.set_ha("right")

fig.suptitle("Projection_hard: EV Charging, Battery Power, and Import Price", fontsize=22)
output_path = PPT_FIGURE_DIR / "ppt_projection_hard_agent_ev_battery_price_15days_clear.png"
if savefigure:
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"Saved {output_path}")
plt.show()


In [7]:
# Diagnostic figure: reward landscape and effective action-space distortion in hard + projection + cost weight
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


savefigure = 1
PROJECT_ROOT = Path(r"D:\RL_Thesis_Project\MADRL_ESS")
PPT_FIGURE_DIR = PROJECT_ROOT / "PPT_Equations" / "PPT_figures"
PPT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RUN_NAME = "madrl_traindays_7"
GAMMA = 0.999
N_STEP = 48

CONTROLLERS = {
    "Base_hard": {"key": "MADRL_BASE_EV_HARD_Continuous", "color": "#4C78A8"},
    "Projection_Costweight_hard": {"key": "MADRL_PROJECTION_EV_HARD_CostWeight", "color": "#F58518"},
}

def read_controller_record(controller_key: str) -> dict:
    record_dir = PROJECT_ROOT / "artifacts" / "runs" / RUN_NAME / "results" / "lstm" / controller_key / "record"
    return {"step": pd.read_parquet(record_dir / "step.parquet").copy(), "agent": pd.read_parquet(record_dir / "agent.parquet").copy()}

def discounted_n_step_return(rewards: np.ndarray, gamma: float = GAMMA, n_step: int = N_STEP) -> np.ndarray:
    out = np.full(len(rewards), np.nan, dtype=float)
    discounts = gamma ** np.arange(n_step)
    for i in range(len(rewards)):
        horizon = min(n_step, len(rewards) - i)
        out[i] = float(np.dot(discounts[:horizon], rewards[i:i + horizon]))
    return out

def add_return_proxy(step: pd.DataFrame) -> pd.DataFrame:
    step = step.sort_values(["episode_idx", "step"]).copy()
    pieces = []
    for _, episode_df in step.groupby("episode_idx", sort=False):
        episode_df = episode_df.copy()
        episode_df["nstep_return_proxy"] = discounted_n_step_return(episode_df["reward_total"].to_numpy(dtype=float))
        pieces.append(episode_df)
    return pd.concat(pieces, ignore_index=True)

def binned_surface(df: pd.DataFrame, x_col: str, y_col: str, value_col: str, x_bins: np.ndarray, y_bins: np.ndarray) -> np.ndarray:
    work = df[[x_col, y_col, value_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    work["x_bin"] = np.digitize(work[x_col].to_numpy(), x_bins) - 1
    work["y_bin"] = np.digitize(work[y_col].to_numpy(), y_bins) - 1
    work = work.loc[work["x_bin"].between(0, len(x_bins) - 2) & work["y_bin"].between(0, len(y_bins) - 2)]
    surface = np.full((len(y_bins) - 1, len(x_bins) - 1), np.nan, dtype=float)
    for (y_idx, x_idx), value in work.groupby(["y_bin", "x_bin"], observed=False)[value_col].mean().items():
        surface[int(y_idx), int(x_idx)] = float(value)
    return surface

def line_profile(step: pd.DataFrame, bins: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, float]:
    work = step[["battery_power_kw", "nstep_return_proxy"]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    work["bin"] = pd.cut(work["battery_power_kw"], bins, include_lowest=True)
    grouped = work.groupby("bin", observed=False)["nstep_return_proxy"]
    median = grouped.median().to_numpy(dtype=float)
    q25 = grouped.quantile(0.25).to_numpy(dtype=float)
    q75 = grouped.quantile(0.75).to_numpy(dtype=float)
    centers = (bins[:-1] + bins[1:]) / 2
    valid = np.isfinite(median)
    if valid.sum() >= 3:
        second_diff = np.diff(median[valid], n=2)
        roughness = float(np.nanmean(np.abs(second_diff)) / (np.nanstd(median[valid]) + 1e-9))
    else:
        roughness = np.nan
    return centers, median, q25, q75, roughness

records = {label: read_controller_record(spec["key"]) for label, spec in CONTROLLERS.items()}
for label in records:
    records[label]["step"] = add_return_proxy(records[label]["step"])

base_step = records["Base_hard"]["step"]
cw_step = records["Projection_Costweight_hard"]["step"]
cw_agent = records["Projection_Costweight_hard"]["agent"].copy()
cw_agent["ev_action_correction_kw"] = cw_agent["ev_charge_kw_executed"] - cw_agent["ev_charge_kw_requested"]
cw_agent["ev_request_bin"] = pd.cut(cw_agent["ev_charge_kw_requested"], np.linspace(0, 11, 18), include_lowest=True)
cw_agent["ev_required_bin"] = pd.cut(cw_agent["ev_required_min_charge_kw"], np.linspace(0, 11, 18), include_lowest=True)
correction_grid = cw_agent.groupby(["ev_required_bin", "ev_request_bin"], observed=False)["ev_action_correction_kw"].mean().unstack().to_numpy(dtype=float)

x_min = min(base_step["battery_power_kw"].quantile(0.01), cw_step["battery_power_kw"].quantile(0.01))
x_max = max(base_step["battery_power_kw"].quantile(0.99), cw_step["battery_power_kw"].quantile(0.99))
y_max = max(base_step["ev_charge_total_requested"].quantile(0.99), cw_step["ev_charge_total_requested"].quantile(0.99), 1.0)
x_bins = np.linspace(x_min, x_max, 28)
y_bins = np.linspace(0, y_max, 24)
base_surface = binned_surface(base_step, "battery_power_kw", "ev_charge_total_requested", "reward_total", x_bins, y_bins)
cw_surface = binned_surface(cw_step, "battery_power_kw", "ev_charge_total_requested", "reward_total", x_bins, y_bins)
finite_surface = np.concatenate([base_surface[np.isfinite(base_surface)], cw_surface[np.isfinite(cw_surface)]])
if len(finite_surface):
    vmin, vmax = np.nanpercentile(finite_surface, [5, 95])
else:
    reward_values = pd.concat([base_step["reward_total"], cw_step["reward_total"]]).replace([np.inf, -np.inf], np.nan).dropna()
    vmin, vmax = np.nanpercentile(reward_values, [5, 95])
line_bins = np.linspace(x_min, x_max, 34)

plt.rcParams.update({"font.size": 15, "axes.titlesize": 17, "axes.labelsize": 15, "legend.fontsize": 12, "xtick.labelsize": 12, "ytick.labelsize": 12})
fig, axes = plt.subplots(2, 2, figsize=(16, 11), constrained_layout=True)

ax = axes[0, 0]
im = ax.imshow(correction_grid, origin="lower", aspect="auto", extent=[0, 11, 0, 11], cmap="OrRd")
ax.plot([0, 11], [0, 11], color="black", linewidth=1.4, linestyle="--", label="request = required minimum")
ax.set_title("A. Hard EV correction compresses action space")
ax.set_xlabel("Requested EV charging per agent (kW)")
ax.set_ylabel("Required minimum charging (kW)")
ax.legend(loc="upper left", frameon=False)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Executed - requested EV power (kW)")

for ax, title, surface in [
    (axes[0, 1], "B. Base_hard immediate reward surface", base_surface),
    (axes[1, 0], "C. Projection_Costweight_hard reward surface", cw_surface),
]:
    mesh = ax.pcolormesh(x_bins, y_bins, surface, cmap="viridis", shading="auto", vmin=vmin, vmax=vmax)
    ax.axvline(0, color="white", linewidth=1.0, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel("System battery power (kW, + charge / - discharge)")
    ax.set_ylabel("Requested EV charging total (kW)")
    cbar = fig.colorbar(mesh, ax=ax)
    cbar.set_label("Mean immediate reward")

ax = axes[1, 1]
for label, spec in CONTROLLERS.items():
    centers, median, q25, q75, roughness = line_profile(records[label]["step"], line_bins)
    color = spec["color"]
    ax.plot(centers, median, color=color, linewidth=2.5, label=f"{label} (roughness={roughness:.2f})")
    ax.fill_between(centers, q25, q75, color=color, alpha=0.16, linewidth=0)
ax.axvline(0, color="black", linewidth=1.0, linestyle="--")
ax.set_title("D. Empirical Q-target proxy vs battery action")
ax.set_xlabel("System battery power (kW, + charge / - discharge)")
ax.set_ylabel(f"{N_STEP}-step discounted return proxy")
ax.legend(frameon=False)

fig.suptitle("Reward and Action-Landscape Diagnostics for Projection_Costweight_hard", fontsize=21, y=1.02)
out = PPT_FIGURE_DIR / "ppt_hard_projection_costweight_learning_landscape_diagnostic.png"
if savefigure:
    fig.savefig(out, dpi=300, bbox_inches="tight")
    print(f"Saved {out}")
plt.show()

Saved D:\RL_Thesis_Project\MADRL_ESS\PPT_Equations\PPT_figures\ppt_hard_projection_costweight_learning_landscape_diagnostic.png


C:\Users\20539\AppData\Local\Temp\ipykernel_21208\4259259865.py:140: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [ ]:
# Split diagnostic figures: hard action correction, reward surface comparison, and Q-target proxy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


savefigure = 1
PROJECT_ROOT = Path(r"D:\RL_Thesis_Project\MADRL_ESS")
PPT_FIGURE_DIR = PROJECT_ROOT / "PPT_Equations" / "PPT_figures"
PPT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RUN_NAME = "madrl_traindays_7"
GAMMA = 0.999
N_STEP = 48

CONTROLLERS = {
    "Base_hard": {"key": "MADRL_BASE_EV_HARD_Continuous", "color": "#4C78A8"},
    "Projection_Costweight_hard": {"key": "MADRL_PROJECTION_EV_HARD_CostWeight", "color": "#F58518"},
}

def read_controller_record(controller_key: str) -> dict:
    record_dir = PROJECT_ROOT / "artifacts" / "runs" / RUN_NAME / "results" / "lstm" / controller_key / "record"
    return {"step": pd.read_parquet(record_dir / "step.parquet").copy(), "agent": pd.read_parquet(record_dir / "agent.parquet").copy()}

def discounted_n_step_return(rewards: np.ndarray, gamma: float = GAMMA, n_step: int = N_STEP) -> np.ndarray:
    out = np.full(len(rewards), np.nan, dtype=float)
    discounts = gamma ** np.arange(n_step)
    for i in range(len(rewards)):
        horizon = min(n_step, len(rewards) - i)
        out[i] = float(np.dot(discounts[:horizon], rewards[i:i + horizon]))
    return out

def add_return_proxy(step: pd.DataFrame) -> pd.DataFrame:
    step = step.sort_values(["episode_idx", "step"]).copy()
    pieces = []
    for _, episode_df in step.groupby("episode_idx", sort=False):
        episode_df = episode_df.copy()
        episode_df["nstep_return_proxy"] = discounted_n_step_return(episode_df["reward_total"].to_numpy(dtype=float))
        pieces.append(episode_df)
    return pd.concat(pieces, ignore_index=True)

def binned_surface(df: pd.DataFrame, x_col: str, y_col: str, value_col: str, x_bins: np.ndarray, y_bins: np.ndarray) -> np.ndarray:
    work = df[[x_col, y_col, value_col]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    work["x_bin"] = np.digitize(work[x_col].to_numpy(), x_bins) - 1
    work["y_bin"] = np.digitize(work[y_col].to_numpy(), y_bins) - 1
    work = work.loc[work["x_bin"].between(0, len(x_bins) - 2) & work["y_bin"].between(0, len(y_bins) - 2)]
    surface = np.full((len(y_bins) - 1, len(x_bins) - 1), np.nan, dtype=float)
    for (y_idx, x_idx), value in work.groupby(["y_bin", "x_bin"], observed=False)[value_col].mean().items():
        surface[int(y_idx), int(x_idx)] = float(value)
    return surface

def line_profile(step: pd.DataFrame, bins: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, float]:
    work = step[["battery_power_kw", "nstep_return_proxy"]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    work["bin"] = pd.cut(work["battery_power_kw"], bins, include_lowest=True)
    grouped = work.groupby("bin", observed=False)["nstep_return_proxy"]
    median = grouped.median().to_numpy(dtype=float)
    q25 = grouped.quantile(0.25).to_numpy(dtype=float)
    q75 = grouped.quantile(0.75).to_numpy(dtype=float)
    centers = (bins[:-1] + bins[1:]) / 2
    valid = np.isfinite(median)
    if valid.sum() >= 3:
        second_diff = np.diff(median[valid], n=2)
        roughness = float(np.nanmean(np.abs(second_diff)) / (np.nanstd(median[valid]) + 1e-9))
    else:
        roughness = np.nan
    return centers, median, q25, q75, roughness

records = {label: read_controller_record(spec["key"]) for label, spec in CONTROLLERS.items()}
for label in records:
    records[label]["step"] = add_return_proxy(records[label]["step"])

base_step = records["Base_hard"]["step"]
cw_step = records["Projection_Costweight_hard"]["step"]
cw_agent = records["Projection_Costweight_hard"]["agent"].copy()
cw_agent["ev_action_correction_kw"] = cw_agent["ev_charge_kw_executed"] - cw_agent["ev_charge_kw_requested"]
cw_agent["ev_request_bin"] = pd.cut(cw_agent["ev_charge_kw_requested"], np.linspace(0, 11, 18), include_lowest=True)
cw_agent["ev_required_bin"] = pd.cut(cw_agent["ev_required_min_charge_kw"], np.linspace(0, 11, 18), include_lowest=True)
correction_grid = cw_agent.groupby(["ev_required_bin", "ev_request_bin"], observed=False)["ev_action_correction_kw"].mean().unstack().to_numpy(dtype=float)

x_min = min(base_step["battery_power_kw"].quantile(0.01), cw_step["battery_power_kw"].quantile(0.01))
x_max = max(base_step["battery_power_kw"].quantile(0.99), cw_step["battery_power_kw"].quantile(0.99))
y_max = max(base_step["ev_charge_total_requested"].quantile(0.99), cw_step["ev_charge_total_requested"].quantile(0.99), 1.0)
x_bins = np.linspace(x_min, x_max, 28)
y_bins = np.linspace(0, y_max, 24)
base_surface = binned_surface(base_step, "battery_power_kw", "ev_charge_total_requested", "reward_total", x_bins, y_bins)
cw_surface = binned_surface(cw_step, "battery_power_kw", "ev_charge_total_requested", "reward_total", x_bins, y_bins)
finite_surface = np.concatenate([base_surface[np.isfinite(base_surface)], cw_surface[np.isfinite(cw_surface)]])
if len(finite_surface):
    vmin, vmax = np.nanpercentile(finite_surface, [5, 95])
else:
    reward_values = pd.concat([base_step["reward_total"], cw_step["reward_total"]]).replace([np.inf, -np.inf], np.nan).dropna()
    vmin, vmax = np.nanpercentile(reward_values, [5, 95])
line_bins = np.linspace(x_min, x_max, 34)

plt.rcParams.update({"font.size": 17, "axes.titlesize": 19, "axes.labelsize": 17, "legend.fontsize": 14, "xtick.labelsize": 14, "ytick.labelsize": 14})

fig, ax = plt.subplots(figsize=(8.4, 6.3), constrained_layout=True)
im = ax.imshow(correction_grid, origin="lower", aspect="auto", extent=[0, 11, 0, 11], cmap="OrRd")
ax.plot([0, 11], [0, 11], color="black", linewidth=1.5, linestyle="--", label="request = required minimum")
ax.set_title("Hard EV Correction Compresses Action Space")
ax.set_xlabel("Requested EV charging per agent (kW)")
ax.set_ylabel("Required minimum charging (kW)")
ax.legend(loc="upper left", frameon=False)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Executed - requested EV power (kW)")
out = PPT_FIGURE_DIR / "ppt_hard_costweight_A_ev_action_space_correction.png"
if savefigure:
    fig.savefig(out, dpi=300, bbox_inches="tight")
    print(f"Saved {out}")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(15.8, 5.9), constrained_layout=True)
mesh = None
for ax, title, surface in [
    (axes[0], "Base_hard Immediate Reward Surface", base_surface),
    (axes[1], "Projection_Costweight_hard Reward Surface", cw_surface),
]:
    mesh = ax.pcolormesh(x_bins, y_bins, surface, cmap="viridis", shading="auto", vmin=vmin, vmax=vmax)
    ax.axvline(0, color="white", linewidth=1.0, alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel("System battery power (kW, + charge / - discharge)")
    ax.set_ylabel("Requested EV charging total (kW)")
cbar = fig.colorbar(mesh, ax=axes.ravel().tolist(), shrink=0.94)
cbar.set_label("Mean immediate reward")
out = PPT_FIGURE_DIR / "ppt_hard_costweight_BC_reward_surface_comparison.png"
if savefigure:
    fig.savefig(out, dpi=300, bbox_inches="tight")
    print(f"Saved {out}")
plt.show()

fig, ax = plt.subplots(figsize=(8.9, 6.0), constrained_layout=True)
for label, spec in CONTROLLERS.items():
    centers, median, q25, q75, roughness = line_profile(records[label]["step"], line_bins)
    color = spec["color"]
    ax.plot(centers, median, color=color, linewidth=2.8, label=f"{label} (roughness={roughness:.2f})")
    ax.fill_between(centers, q25, q75, color=color, alpha=0.16, linewidth=0)
ax.axvline(0, color="black", linewidth=1.1, linestyle="--")
ax.set_title("Empirical Q-target Proxy vs Battery Action")
ax.set_xlabel("System battery power (kW, + charge / - discharge)")
ax.set_ylabel(f"{N_STEP}-step discounted return proxy")
ax.legend(frameon=False, loc="upper right")
out = PPT_FIGURE_DIR / "ppt_hard_costweight_D_q_target_proxy_battery_action.png"
if savefigure:
    fig.savefig(out, dpi=300, bbox_inches="tight")
    print(f"Saved {out}")
plt.show()


In [ ]:
# High-price EV charging share: Projection_Costweight_hard vs Projection_Priceaware_hard
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


savefigure = 1
PROJECT_ROOT = Path(r"D:\RL_Thesis_Project\MADRL_ESS")
PPT_FIGURE_DIR = PROJECT_ROOT / "PPT_Equations" / "PPT_figures"
PPT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RUN_NAME = "madrl_traindays_7"
DT_HOURS = 0.25
PRICE_QUANTILE = 0.70

CONTROLLERS = {
    "Base_hard": {
        "key": "MADRL_BASE_EV_HARD_Continuous",
        "color": "#54A24B",
    },
    "Projection_Costweight_hard": {
        "key": "MADRL_PROJECTION_EV_HARD_CostWeight",
        "color": "#4C78A8",
    },
    "Projection_Priceaware_hard": {
        "key": "MADRL_PROJECTION_EV_HARD_CostWeight_PriceAware",
        "color": "#F58518",
    },
}


def local_time(series):
    return pd.to_datetime(series.astype(str).str.replace(r"\+\d{2}:\d{2}$", "", regex=True))


def read_record(controller_key):
    record_dir = PROJECT_ROOT / "artifacts" / "runs" / RUN_NAME / "results" / "lstm" / controller_key / "record"
    step = pd.read_parquet(record_dir / "step.parquet").copy()
    agent = pd.read_parquet(record_dir / "agent.parquet").copy()
    step["timestamp"] = local_time(step["timestamp"])
    agent["timestamp"] = local_time(agent["timestamp"])
    return step, agent


def high_price_ev_summary(label, controller_key):
    step, agent = read_record(controller_key)
    price_threshold = step.groupby("episode_idx")["import_price"].quantile(PRICE_QUANTILE).rename("high_price_threshold")
    price_lookup = step[["episode_idx", "timestamp", "import_price"]].merge(price_threshold, on="episode_idx", how="left")
    merged = agent.merge(price_lookup, on=["episode_idx", "timestamp"], how="left", validate="many_to_one")
    merged["ev_energy_kwh"] = merged["ev_charge_kw"].clip(lower=0.0) * DT_HOURS
    merged["high_price_energy_kwh"] = np.where(
        merged["import_price"] >= merged["high_price_threshold"],
        merged["ev_energy_kwh"],
        0.0,
    )
    total_energy = float(merged["ev_energy_kwh"].sum())
    high_energy = float(merged["high_price_energy_kwh"].sum())
    avg_charge_price = float((merged["ev_energy_kwh"] * merged["import_price"]).sum() / total_energy)
    return {
        "Controller": label,
        "High-price share": high_energy / total_energy,
        "High-price EV energy": high_energy,
        "Total EV energy": total_energy,
        "Avg EV charging price": avg_charge_price,
    }


summary = pd.DataFrame(
    high_price_ev_summary(label, spec["key"])
    for label, spec in CONTROLLERS.items()
)

plt.rcParams.update(
    {
        "font.family": "Times New Roman",
        "axes.titlesize": 22,
        "axes.labelsize": 17,
        "xtick.labelsize": 15,
        "ytick.labelsize": 15,
        "legend.fontsize": 17,
        "axes.linewidth": 1.3,
    }
)

fig, ax = plt.subplots(figsize=(12.2, 5.7))
x = np.arange(len(summary))
colors = [CONTROLLERS[label]["color"] for label in summary["Controller"]]
bars = ax.bar(x, summary["High-price share"] * 100.0, color=colors, width=0.55)

for idx, bar in enumerate(bars):
    share = summary.loc[idx, "High-price share"] * 100.0
    avg_price = summary.loc[idx, "Avg EV charging price"]
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1.0,
        f"{share:.1f}%\nAvg price: {avg_price:.3f}",
        ha="center",
        va="bottom",
        fontsize=14,
    )

ax.set_title("High-price EV Charging Share under Hard Constraint")
ax.set_ylabel("High-price EV charging share (%)")
ax.set_xticks(x)
ax.set_xticklabels(["Base_hard", "Projection_\nCostweight_hard", "Projection_\nPriceaware_hard"], rotation=0, ha="center")
ax.set_ylim(0, max(45.0, float((summary["High-price share"] * 100.0).max()) + 9.0))
ax.grid(axis="y", alpha=0.28, linewidth=1.0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

caption = "High-price period is defined separately within each episode using the 70% price quantile."
fig.text(0.5, 0.02, caption, ha="center", va="bottom", fontsize=13, color="#23364A")
fig.subplots_adjust(left=0.115, right=0.98, top=0.88, bottom=0.24)

out_path = PPT_FIGURE_DIR / "ppt_hard_priceaware_high_price_ev_charging_share.png"
if savefigure:
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close(fig)

print(summary)
print(out_path)


In [ ]:
# Price-aware penalty mechanism: price threshold and penalized EV charging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.dates as mdates
import matplotlib.pyplot as plt


savefigure = 1
PROJECT_ROOT = Path(r"D:\RL_Thesis_Project\MADRL_ESS")
PPT_FIGURE_DIR = PROJECT_ROOT / "PPT_Equations" / "PPT_figures"
PPT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RUN_NAME = "madrl_traindays_7"
CONTROLLER_KEY = "MADRL_PROJECTION_EV_HARD_CostWeight_PriceAware"
WINDOW_START = pd.Timestamp("2020-04-02 18:00")
WINDOW_END = pd.Timestamp("2020-04-03 07:00")


def local_time(series):
    return pd.to_datetime(series.astype(str).str.replace(r"\+\d{2}:\d{2}$", "", regex=True))


record_dir = PROJECT_ROOT / "artifacts" / "runs" / RUN_NAME / "results" / "lstm" / CONTROLLER_KEY / "record"
step = pd.read_parquet(record_dir / "step.parquet").copy()
agent = pd.read_parquet(record_dir / "agent.parquet").copy()
step["timestamp"] = local_time(step["timestamp"])
agent["timestamp"] = local_time(agent["timestamp"])

window_step = step.loc[(step["timestamp"] >= WINDOW_START) & (step["timestamp"] <= WINDOW_END)].sort_values("timestamp").copy()
window_agent = agent.loc[(agent["timestamp"] >= WINDOW_START) & (agent["timestamp"] <= WINDOW_END)].copy()
ev_power = window_agent.groupby("timestamp")["ev_charge_kw"].sum().reindex(window_step["timestamp"], fill_value=0.0)
ev_available = window_agent.groupby("timestamp")["ev_available"].sum().reindex(window_step["timestamp"], fill_value=0.0)
threshold = window_step["ev_price_aware_threshold"].ffill().bfill()
high_price = window_step["import_price"].to_numpy() >= threshold.to_numpy()
penalized_ev = np.where(high_price, ev_power.to_numpy(), 0.0)

plt.rcParams.update(
    {
        "font.family": "Times New Roman",
        "axes.titlesize": 22,
        "axes.labelsize": 17,
        "xtick.labelsize": 15,
        "ytick.labelsize": 15,
        "legend.fontsize": 15,
        "axes.linewidth": 1.25,
    }
)

fig, axes = plt.subplots(2, 1, figsize=(13.5, 7.2), sharex=True, gridspec_kw={"height_ratios": [1.0, 1.25], "hspace": 0.16})

axes[0].plot(window_step["timestamp"], window_step["import_price"], color="black", linewidth=2.4, label="Import price")
axes[0].plot(window_step["timestamp"], threshold, color="#D62728", linestyle="--", linewidth=2.0, label="70% price threshold")
axes[0].fill_between(
    window_step["timestamp"],
    threshold,
    window_step["import_price"],
    where=high_price,
    color="#F28E8C",
    alpha=0.35,
    interpolate=True,
    label="High-price period",
)
axes[0].set_title("Price-aware Penalty Mechanism during EV Connected Window")
axes[0].set_ylabel("Import price")
axes[0].legend(loc="upper left", frameon=False, ncol=3)
axes[0].grid(True, alpha=0.28, linewidth=1.0)

axes[1].fill_between(
    window_step["timestamp"],
    0,
    ev_available * max(float(ev_power.max()), 1.0) / max(float(ev_available.max()), 1.0),
    where=ev_available.to_numpy() > 0,
    color="#DDEAF7",
    alpha=0.75,
    step="mid",
    label="EV connected window",
)
axes[1].plot(window_step["timestamp"], ev_power, color="#4C78A8", linewidth=2.8, label="Total EV charging")
axes[1].fill_between(window_step["timestamp"], 0, penalized_ev, color="#D62728", alpha=0.30, label="EV charging with extra penalty")
axes[1].set_ylabel("EV charging power (kW)")
axes[1].set_xlabel("Time")
axes[1].legend(loc="upper left", frameon=False, ncol=3)
axes[1].grid(True, alpha=0.28, linewidth=1.0)
axes[1].xaxis.set_major_locator(mdates.HourLocator(byhour=range(0, 24, 2)))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H"))
axes[1].tick_params(axis="x", rotation=30)
axes[1].set_xlim(WINDOW_START, WINDOW_END)

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.subplots_adjust(left=0.085, right=0.985, top=0.91, bottom=0.16)

out_path = PPT_FIGURE_DIR / "ppt_priceaware_penalty_mechanism_ev_charge_threshold.png"
if savefigure:
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"Window: {WINDOW_START} to {WINDOW_END}")
print(f"Mean threshold: {threshold.mean():.4f}")
print(out_path)


In [ ]:
# PPT refinement: overnight EV and signed battery competition under Base_Costweight_soft
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.dates as mdates
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import pandas as pd


savefigure = 1
PROJECT_ROOT = Path(r"D:\RL_Thesis_Project\MADRL_ESS")
PPT_FIGURE_DIR = PROJECT_ROOT / "PPT_Equations" / "PPT_figures"
PPT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
RUN_NAME = "madrl_traindays_7"
CONTROLLER_KEY = "MADRL_BASE_COSTWEIGHT"
CONTROLLER_LABEL = "Cost-Weighted Soft"
EPISODE_TO_PLOT = 11
AGENT_TO_PLOT = 2
DT_HOURS = 0.25


def local_time(series):
    return pd.to_datetime(series.astype(str).str.replace(r"\+\d{2}:\d{2}$", "", regex=True))


record_dir = PROJECT_ROOT / "artifacts" / "runs" / RUN_NAME / "results" / "lstm" / CONTROLLER_KEY / "record"
step = pd.read_parquet(record_dir / "step.parquet").copy()
agent = pd.read_parquet(record_dir / "agent.parquet").copy()
step["timestamp"] = local_time(step["timestamp"])
agent["timestamp"] = local_time(agent["timestamp"])

episode_start = step.loc[step["episode_idx"] == EPISODE_TO_PLOT, "timestamp"].min().normalize()
window_start = episode_start + pd.Timedelta(hours=18)
window_end = episode_start + pd.Timedelta(days=1, hours=7)

window_step = step.loc[(step["timestamp"] >= window_start) & (step["timestamp"] <= window_end)].sort_values("timestamp").copy()
window_agent = agent.loc[
    (agent["agent_id"] == AGENT_TO_PLOT)
    & (agent["timestamp"] >= window_start)
    & (agent["timestamp"] <= window_end)
].sort_values("timestamp").copy()
if window_step.empty or window_agent.empty:
    raise ValueError(f"No overnight records found for {CONTROLLER_LABEL}, agent {AGENT_TO_PLOT}.")

plot_df = window_agent.merge(window_step[["timestamp", "import_price"]], on="timestamp", how="left")
low_price_threshold = window_step["import_price"].quantile(0.25)
low_price = window_step["import_price"] <= low_price_threshold

plt.rcParams.update(
    {
        "font.family": "Times New Roman",
        "axes.titlesize": 18,
        "axes.labelsize": 16,
        "xtick.labelsize": 14,
        "ytick.labelsize": 14,
        "legend.fontsize": 12.5,
        "axes.linewidth": 1.15,
    }
)

fig, ax = plt.subplots(figsize=(12.8, 6.2))
price_ax = ax.twinx()

for idx in window_step.index[low_price]:
    start = window_step.loc[idx, "timestamp"]
    next_rows = window_step.loc[window_step["timestamp"] > start, "timestamp"]
    end = next_rows.iloc[0] if len(next_rows) else start + pd.Timedelta(minutes=15)
    ax.axvspan(start, end, color="#D7E9FF", alpha=0.45, linewidth=0)

competition_ts = set(window_step.loc[low_price, "timestamp"])
competition_mask = (plot_df["timestamp"].isin(competition_ts)) & (plot_df["ev_charge_kw"] > 1.0) & (plot_df["e_bat"] > 5.0)
for ts in plot_df.loc[competition_mask, "timestamp"]:
    ax.axvspan(ts, ts + pd.Timedelta(minutes=15), color="#F4A6A6", alpha=0.45, linewidth=0)

ax.axhline(0.0, color="#1F2933", linewidth=1.0, alpha=0.65)
ax.plot(plot_df["timestamp"], plot_df["ev_charge_kw"], color="#4C78A8", linewidth=2.3, label="EV charging")
ax.fill_between(plot_df["timestamp"], 0, plot_df["ev_charge_kw"], color="#4C78A8", alpha=0.18)
ax.plot(plot_df["timestamp"], plot_df["e_bat"], color="#F58518", linewidth=2.2, label="Battery power (+ charge / - discharge)")
ax.fill_between(plot_df["timestamp"], 0, plot_df["e_bat"], where=plot_df["e_bat"] >= 0, color="#F58518", alpha=0.18, interpolate=True)
ax.fill_between(plot_df["timestamp"], 0, plot_df["e_bat"], where=plot_df["e_bat"] < 0, color="#54A24B", alpha=0.18, interpolate=True)

price_ax.plot(window_step["timestamp"], window_step["import_price"], color="black", linestyle="--", linewidth=1.45, label="Import price")
price_ax.axhline(low_price_threshold, color="black", linewidth=1.0, alpha=0.4, linestyle=":", label="Low-price threshold")

ax.set_title(f"{CONTROLLER_LABEL}: EV and Battery Compete for Low-Price Periods (Agent {AGENT_TO_PLOT}, Apr 12-13)")
ax.set_ylabel("Power (kW)")
ax.set_xlabel("")
price_ax.set_ylabel("Import price")
ax.grid(True, alpha=0.25, linewidth=0.9)
ax.set_xlim(window_start, window_end)
ax.xaxis.set_major_locator(mdates.HourLocator(byhour=range(0, 24, 2)))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
ax.tick_params(axis="x", rotation=30)

lines, labels = ax.get_legend_handles_labels()
price_lines, price_labels = price_ax.get_legend_handles_labels()
extra_handles = [Patch(facecolor="#D7E9FF", alpha=0.45, label="Low-price period"), Patch(facecolor="#F4A6A6", alpha=0.45, label="EV-battery charging overlap")]
ax.legend(lines + price_lines + extra_handles, labels + price_labels + [h.get_label() for h in extra_handles], loc="upper center", bbox_to_anchor=(0.5, -0.30), frameon=False, ncol=3, borderaxespad=0.0)

for spine in ["top"]:
    ax.spines[spine].set_visible(False)
    price_ax.spines[spine].set_visible(False)

fig.tight_layout(rect=(0, 0.20, 1, 1))
out_path = PPT_FIGURE_DIR / "ppt_costweighted_soft_agent2_overnight_ev_battery_competition_large_fonts.png"
thesis_out_path = THESIS_FIGURE_DIR / "ch6_3_costweighted_soft_low_price_competition.png"
if savefigure:
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    fig.savefig(thesis_out_path, dpi=300, bbox_inches="tight")
plt.close(fig)

print(f"Saved figures to: {out_path} and {thesis_out_path}")
print(f"Window: {window_start} to {window_end}")
print(f"Low-price threshold: {low_price_threshold:.4f}")


In [ ]:
# Modified: Overnight comparison for the Projected Hard SoC-upper-limit failure analysis.
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path(r"D:\\RL_Thesis_Project\\MADRL_ESS")
RECORD_ROOT = PROJECT_ROOT / "artifacts" / "runs" / "madrl_traindays_7" / "results" / "lstm"
THESIS_FIGURE_DIR = PROJECT_ROOT / "AAA-Thesis" / "Thesis" / "figures"
THESIS_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
CONTROLLERS = [("MADRL_PROJECTION_EV_HARD_Continuous2", "Projected Hard"), ("MADRL_PROJECTION_EV_HARD_Continuous3", "Projected Hard (EV SoC Upper Limit 0.91)")]
WINDOW_START = pd.Timestamp("2020-04-01 18:00:00")
WINDOW_END = pd.Timestamp("2020-04-02 07:00:00")
AGENT_IDS = [0, 1, 2]
DT_HOURS = 0.25

def local_time(series):
    return pd.to_datetime(series.astype(str).str.replace(r"\+\d{2}:\d{2}$", "", regex=True))

records = {}
for key, label in CONTROLLERS:
    record_dir = RECORD_ROOT / key / "record"
    agent = pd.read_parquet(record_dir / "agent.parquet").copy()
    step = pd.read_parquet(record_dir / "step.parquet").copy()
    agent["timestamp"] = local_time(agent["timestamp"])
    step["timestamp"] = local_time(step["timestamp"])
    records[label] = {"agent": agent.loc[agent["timestamp"].between(WINDOW_START, WINDOW_END)].copy(), "step": step.loc[step["timestamp"].between(WINDOW_START, WINDOW_END)].copy()}

plt.rcParams.update({"font.family": "Times New Roman", "axes.titlesize": 13, "axes.labelsize": 11, "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 10})

def plot_overnight(variable, ylabel, filename, colors):
    fig, axes = plt.subplots(len(AGENT_IDS), len(CONTROLLERS), figsize=(13.2, 8.4), sharex=True, sharey=True)
    for row, agent_id in enumerate(AGENT_IDS):
        for col, (_, label) in enumerate(CONTROLLERS):
            ax = axes[row, col]
            price_ax = ax.twinx()
            one = records[label]["agent"].loc[records[label]["agent"]["agent_id"] == agent_id].sort_values("timestamp")
            price = records[label]["step"].sort_values("timestamp")
            ax.axhline(0, color="#777777", linewidth=0.7)
            ax.plot(one["timestamp"], one[variable], color=colors[col], linewidth=1.8)
            price_ax.plot(price["timestamp"], price["import_price"], color="black", linestyle="--", linewidth=1.0, alpha=0.8)
            ax.set_title(f"{label} -- Agent {agent_id}")
            if col == 0:
                ax.set_ylabel(ylabel)
            else:
                price_ax.set_ylabel("Import price")
            ax.grid(True, alpha=0.22)
            ax.xaxis.set_major_locator(mdates.HourLocator(interval=2))
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
            ax.tick_params(axis="x", rotation=25)
            ax.spines["top"].set_visible(False)
            price_ax.spines["top"].set_visible(False)
    fig.legend([plt.Line2D([0], [0], color=colors[0], linewidth=2), plt.Line2D([0], [0], color="black", linestyle="--")], [ylabel, "Import price"], loc="lower center", ncol=2, frameon=False)
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    output_path = THESIS_FIGURE_DIR / filename
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure to: {output_path}")

plot_overnight("ev_charge_kw", "EV charging power (kW)", "ch6_6_soc091_overnight_ev_price_comparison.png", ["#4C78A8", "#E45756"])
plot_overnight("e_bat", "Battery power (kW)", "ch6_6_soc091_overnight_battery_price_comparison.png", ["#4C78A8", "#E45756"])

for _, label in CONTROLLERS:
    overnight = records[label]["agent"]
    summary = overnight.groupby("agent_id").agg(ev_energy_kwh=("ev_charge_kw", lambda x: x.clip(lower=0).sum() * DT_HOURS), battery_charge_kwh=("e_bat", lambda x: x.clip(lower=0).sum() * DT_HOURS), battery_discharge_kwh=("e_bat", lambda x: -x.clip(upper=0).sum() * DT_HOURS))
    print(label)
    print(summary.round(2).to_string())
